In [3]:

from pathlib import Path
import pandas as pd
import joblib

# Project paths
PROJECT_DIR = Path(
    r"C:\Users\saksham\Desktop\Project\f1-prediction-platform"
)

DATA_PATH = (
    PROJECT_DIR / "data" / "processed"
    / "expanded_model_dataset.csv"
)

MODEL_DIR = PROJECT_DIR / "models"
REPORT_DIR = PROJECT_DIR / "reports"

# Load the saved dataset
expanded_df = pd.read_csv(DATA_PATH)

# Restore the feature list
expanded_features = [
    "starting_position_feature",
    "driver_prior_mean_finish",
    "driver_prior_win_rate",
    "driver_recent5_mean_finish",
    "constructor_prior_mean_finish",
    "constructor_prior_win_rate",
]

target = "won_race"

# Restore the saved models
baseline_model = joblib.load(
    MODEL_DIR / "baseline_model.joblib"
)

enhanced_model = joblib.load(
    MODEL_DIR / "logistic_regression_model.joblib"
)

rf_model = joblib.load(
    MODEL_DIR / "random_forest_model.joblib"
)

print("Project restored successfully")
print("Dataset shape:", expanded_df.shape)
print("Seasons:", sorted(expanded_df["season"].unique()))
print("Models: Baseline, Logistic Regression, Random Forest")


Project restored successfully
Dataset shape: (3458, 27)
Seasons: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Models: Baseline, Logistic Regression, Random Forest



# F1 Race Prediction Platform

## Notebook 01: Data Collection

### Objective
Build a reliable historical and upcoming-race dataset
for an F1 race winner prediction system.

### Data sources
- Jolpica F1 API: historical results and race schedules
- Additional sources later: qualifying, weather and timing data

### Workflow
1. Connect to the API
2. Retrieve the race calendar
3. Collect historical race and qualifying results
4. Validate and save the raw datasets

### Important rule
Every prediction must use only information available
at its prediction cutoff.

In [4]:
from pathlib import Path

import pandas as pd
import requests
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 30)

# Locate the project root
cwd = Path.cwd()
PROJECT_DIR = cwd.parent if cwd.name == "notebooks" else cwd

# Data directories
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR.resolve())
print("Raw data directory:", RAW_DIR.resolve())
print("Processed data directory:", PROCESSED_DIR.resolve())

Project directory: C:\Users\saksham\Desktop\Project\f1-prediction-platform
Raw data directory: C:\Users\saksham\Desktop\Project\f1-prediction-platform\data\raw
Processed data directory: C:\Users\saksham\Desktop\Project\f1-prediction-platform\data\processed


In [5]:
BASE_URL = "https://api.jolpi.ca/ergast/f1"

response = requests.get(
    f"{BASE_URL}/2025/races.json",
    params={"limit": 100, "format": "json"},
    timeout=30,
)

print("HTTP status:", response.status_code)

response.raise_for_status()

payload = response.json()
race_data = payload["MRData"]["RaceTable"]["Races"]

print("Number of races returned:", len(race_data))

if race_data:
    print("First race:", race_data[0]["raceName"])
    print("First race date:", race_data[0]["date"])

HTTP status: 200
Number of races returned: 24
First race: Australian Grand Prix
First race date: 2025-03-16


In [6]:
import time

SEASONS = range(2018, 2027)
API_LIMIT = 100


def fetch_all_races(endpoint):
    """Fetch every page of race records from a Jolpica endpoint."""
    all_races = []
    offset = 0

    while True:
        response = requests.get(
            f"{BASE_URL}/{endpoint}",
            params={
                "limit": API_LIMIT,
                "offset": offset,
                "format": "json",
            },
            timeout=30,
        )
        response.raise_for_status()

        mr_data = response.json()["MRData"]
        race_table = mr_data["RaceTable"]
        races = race_table.get("Races", [])
        total = int(mr_data.get("total", 0))

        all_races.extend(races)
        offset += len(races)

        if not races or offset >= total:
            break

        time.sleep(0.2)

    return all_races


calendar_records = []

for season in SEASONS:
    races = fetch_all_races(f"{season}/races.json")

    for race in races:
        circuit = race.get("Circuit", {})
        location = circuit.get("Location", {})

        calendar_records.append({
            "season": int(race["season"]),
            "round": int(race["round"]),
            "race_name": race["raceName"],
            "race_date": race.get("date"),
            "race_time_utc": race.get("time"),
            "circuit_id": circuit.get("circuitId"),
            "circuit_name": circuit.get("circuitName"),
            "city": location.get("locality"),
            "country": location.get("country"),
            "latitude": location.get("lat"),
            "longitude": location.get("long"),
            "url": race.get("url"),
        })

    print(f"{season}: {len(races)} calendar entries")

2018: 21 calendar entries
2019: 21 calendar entries
2020: 17 calendar entries
2021: 22 calendar entries
2022: 22 calendar entries
2023: 22 calendar entries
2024: 24 calendar entries
2025: 24 calendar entries
2026: 23 calendar entries


In [7]:
calendar_df = pd.DataFrame(calendar_records)

calendar_df = (
    calendar_df
    .drop_duplicates(subset=["season", "round"])
    .sort_values(["season", "round"])
    .reset_index(drop=True)
)

print("Dataset shape:", calendar_df.shape)
print("Seasons collected:", sorted(calendar_df["season"].unique()))

calendar_df.head(10)

Dataset shape: (196, 12)
Seasons collected: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]


,season,round,race_name,race_date,race_time_utc,circuit_id,circuit_name,city,country,latitude,longitude,url
0,2018,1,Australian Grand Prix,2018-03-25,05:10:00Z,albert_park,Albert Park Grand Prix Circuit,Melbourne,Australia,-37.8497,144.968,https://en.wikipedia.org/wiki/2018_Australian_...
1,2018,2,Bahrain Grand Prix,2018-04-08,15:10:00Z,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,https://en.wikipedia.org/wiki/2018_Bahrain_Gra...
2,2018,3,Chinese Grand Prix,2018-04-15,06:10:00Z,shanghai,Shanghai International Circuit,Shanghai,China,31.3389,121.22,https://en.wikipedia.org/wiki/2018_Chinese_Gra...
3,2018,4,Azerbaijan Grand Prix,2018-04-29,12:10:00Z,baku,Baku City Circuit,Baku,Azerbaijan,40.3725,49.8533,https://en.wikipedia.org/wiki/2018_Azerbaijan_...
4,2018,5,Spanish Grand Prix,2018-05-13,13:10:00Z,catalunya,Circuit de Barcelona-Catalunya,Barcelona,Spain,41.57,2.26111,https://en.wikipedia.org/wiki/2018_Spanish_Gra...
5,2018,6,Monaco Grand Prix,2018-05-27,13:10:00Z,monaco,Circuit de Monaco,Monte Carlo,Monaco,43.7347,7.42056,https://en.wikipedia.org/wiki/2018_Monaco_Gran...
6,2018,7,Canadian Grand Prix,2018-06-10,18:10:00Z,villeneuve,Circuit Gilles Villeneuve,Montreal,Canada,45.5,-73.5228,https://en.wikipedia.org/wiki/2018_Canadian_Gr...
7,2018,8,French Grand Prix,2018-06-24,14:10:00Z,ricard,Circuit Paul Ricard,Le Castellet,France,43.2506,5.79167,https://en.wikipedia.org/wiki/2018_French_Gran...
8,2018,9,Austrian Grand Prix,2018-07-01,13:10:00Z,red_bull_ring,Red Bull Ring,Spielberg,Austria,47.2197,14.7647,https://en.wikipedia.org/wiki/2018_Austrian_Gr...
9,2018,10,British Grand Prix,2018-07-08,13:10:00Z,silverstone,Silverstone Circuit,Silverstone,UK,52.0786,-1.01694,https://en.wikipedia.org/wiki/2018_British_Gra...


In [8]:
print("Missing values:")
display(calendar_df.isna().sum().to_frame("missing_count"))

print("\nDuplicate season-round combinations:")
print(calendar_df.duplicated(["season", "round"]).sum())

print("\nEntries per season:")
display(
    calendar_df.groupby("season")
    .agg(
        scheduled_rounds=("round", "nunique"),
        first_race=("race_date", "min"),
        last_race=("race_date", "max"),
    )
)

Missing values:


,missing_count
season,0
round,0
race_name,0
race_date,0
race_time_utc,0
circuit_id,0
circuit_name,0
city,0
country,0
latitude,0



Duplicate season-round combinations:
0

Entries per season:


,scheduled_rounds,first_race,last_race
season,,,
2018,21,2018-03-25,2018-11-25
2019,21,2019-03-17,2019-12-01
2020,17,2020-07-05,2020-12-13
2021,22,2021-03-28,2021-12-12
2022,22,2022-03-20,2022-11-20
2023,22,2023-03-05,2023-11-26
2024,24,2024-03-02,2024-12-08
2025,24,2025-03-16,2025-12-07
2026,23,2026-03-08,2026-12-06


In [9]:
calendar_path = RAW_DIR / "race_calendar.csv"

calendar_df.to_csv(calendar_path, index=False)

print(f"Saved {len(calendar_df)} calendar entries to:")
print(calendar_path.resolve())

Saved 196 calendar entries to:
C:\Users\saksham\Desktop\Project\f1-prediction-platform\data\raw\race_calendar.csv


In [10]:
from datetime import datetime, timezone

today_utc = datetime.now(timezone.utc).date()

calendar_df["race_date"] = pd.to_datetime(
    calendar_df["race_date"],
    errors="coerce",
).dt.date

upcoming_races = (
    calendar_df[calendar_df["race_date"] >= today_utc]
    .sort_values(["race_date", "season", "round"])
    .reset_index(drop=True)
)

if upcoming_races.empty:
    print("No upcoming races found in the downloaded calendar.")
else:
    display(
        upcoming_races[
            ["season", "round", "race_name", "race_date", "circuit_name", "country"]
        ].head(10)
    )

,season,round,race_name,race_date,circuit_name,country
0,2026,18,United States Grand Prix,2026-10-25,Circuit of the Americas,USA
1,2026,19,Mexico City Grand Prix,2026-11-01,Autódromo Hermanos Rodríguez,Mexico
2,2026,20,Brazilian Grand Prix,2026-11-08,Autódromo José Carlos Pace,Brazil
3,2026,21,Las Vegas Grand Prix,2026-11-22,Las Vegas Strip Street Circuit,USA
4,2026,22,Qatar Grand Prix,2026-11-29,Lusail International Circuit,Qatar
5,2026,23,Abu Dhabi Grand Prix,2026-12-06,Yas Marina Circuit,UAE


In [11]:

import time
from requests.exceptions import RequestException

def get_json_with_retry(url, params=None, max_attempts=7):
    """Request JSON with retries for rate limits and temporary server errors."""
    for attempt in range(max_attempts):
        try:
            response = requests.get(url, params=params, timeout=45)

            if response.status_code == 429 or response.status_code in {
                500, 502, 503, 504
            }:
                retry_after = response.headers.get("Retry-After")
                wait_seconds = (
                    float(retry_after)
                    if retry_after and retry_after.replace(".", "", 1).isdigit()
                    else min(5 * (2 ** attempt), 90)
                )

                if attempt == max_attempts - 1:
                    response.raise_for_status()

                print(
                    f"HTTP {response.status_code}; "
                    f"waiting {wait_seconds:.0f}s before retry..."
                )
                time.sleep(wait_seconds)
                continue

            response.raise_for_status()
            return response.json()

        except RequestException:
            if attempt == max_attempts - 1:
                raise
            time.sleep(min(5 * (2 ** attempt), 90))

    raise RuntimeError(f"Could not fetch {url}")


# Start clean: the previous attempt may contain duplicate records.
results_records = []

for season in SEASONS:
    season_count = 0

    # Use the already collected calendar to identify races for this season.
    season_races = (
        calendar_df[calendar_df["season"] == season]
        .sort_values("round")
    )

    for _, race_row in season_races.iterrows():
        round_number = int(race_row["round"])
        url = f"{BASE_URL}/{season}/{round_number}/results.json"

        payload = get_json_with_retry(
            url,
            params={"limit": 100, "format": "json"}
        )

        races = payload["MRData"]["RaceTable"].get("Races", [])

        for race in races:
            for result in race.get("Results", []):
                driver = result.get("Driver", {})
                constructor = result.get("Constructor", {})
                time_data = result.get("Time", {})
                fastest_lap = result.get("FastestLap", {})

                results_records.append({
                    "season": int(race["season"]),
                    "round": int(race["round"]),
                    "race_name": race["raceName"],
                    "race_date": race.get("date"),
                    "driver_id": driver.get("driverId"),
                    "driver_code": driver.get("code"),
                    "driver_number": driver.get("permanentNumber"),
                    "driver_name": " ".join(filter(None, [
                        driver.get("givenName"),
                        driver.get("familyName")
                    ])),
                    "constructor_id": constructor.get("constructorId"),
                    "constructor_name": constructor.get("name"),
                    "grid": result.get("grid"),
                    "position": result.get("position"),
                    "position_text": result.get("positionText"),
                    "points": result.get("points"),
                    "laps": result.get("laps"),
                    "status": result.get("status"),
                    "finish_time": time_data.get("time"),
                    "fastest_lap_rank": fastest_lap.get("rank"),
                    "fastest_lap_time": fastest_lap.get(
                        "Time", {}
                    ).get("time"),
                })
                season_count += 1

        # Be considerate of the API's request limits.
        time.sleep(1.0)

    print(f"{season}: {season_count} driver-result records")

KeyboardInterrupt: 

In [12]:

results_df = pd.DataFrame(results_records)

results_df = (
    results_df
    .drop_duplicates(subset=["season", "round", "driver_id"])
    .sort_values(["season", "round"])
    .reset_index(drop=True)
)

results_df["position_num"] = pd.to_numeric(
    results_df["position"], errors="coerce"
)

print("Dataset shape:", results_df.shape)
print("Seasons collected:", sorted(results_df["season"].unique()))
print("Unique drivers:", results_df["driver_id"].nunique())
print(
    "Unique races:",
    results_df[["season", "round"]].drop_duplicates().shape[0]
)

display(results_df.head(5))

Dataset shape: (120, 20)
Seasons collected: [np.int64(2018)]
Unique drivers: 20
Unique races: 6


,season,round,race_name,race_date,driver_id,driver_code,driver_number,driver_name,constructor_id,constructor_name,grid,position,position_text,points,laps,status,finish_time,fastest_lap_rank,fastest_lap_time,position_num
0,2018,1,Australian Grand Prix,2018-03-25,vettel,VET,5,Sebastian Vettel,ferrari,Ferrari,3,1,1,25,58,Finished,1:29:33.283,4,1:26.469,1
1,2018,1,Australian Grand Prix,2018-03-25,hamilton,HAM,44,Lewis Hamilton,mercedes,Mercedes,1,2,2,18,58,Finished,+5.036,3,1:26.444,2
2,2018,1,Australian Grand Prix,2018-03-25,raikkonen,RAI,7,Kimi Räikkönen,ferrari,Ferrari,2,3,3,15,58,Finished,+6.309,2,1:26.373,3
3,2018,1,Australian Grand Prix,2018-03-25,ricciardo,RIC,3,Daniel Ricciardo,red_bull,Red Bull,8,4,4,12,58,Finished,+7.069,1,1:25.945,4
4,2018,1,Australian Grand Prix,2018-03-25,alonso,ALO,14,Fernando Alonso,mclaren,McLaren,10,5,5,10,58,Finished,+27.886,7,1:26.978,5


In [13]:

results_df = pd.DataFrame(results_records)

results_df = (
    results_df
    .drop_duplicates(subset=["season", "round", "driver_id"])
    .sort_values(["season", "round"])
    .reset_index(drop=True)
)

results_df["position_num"] = pd.to_numeric(
    results_df["position"],
    errors="coerce"
)

print("Dataset shape:", results_df.shape)
print("Unique drivers:", results_df["driver_id"].nunique())
print(
    "Unique races:",
    results_df[["season", "round"]].drop_duplicates().shape[0]
)

print("\nResults per season:")
display(
    results_df.groupby("season").agg(
        result_records=("driver_id", "size"),
        races=("round", "nunique"),
        last_round=("round", "max"),
        last_race_date=("race_date", "max"),
    )
)

print("\nFirst 5 records:")
display(results_df.head())

Dataset shape: (120, 20)
Unique drivers: 20
Unique races: 6

Results per season:


,result_records,races,last_round,last_race_date
season,,,,
2018,120,6,6,2018-05-27



First 5 records:


,season,round,race_name,race_date,driver_id,driver_code,driver_number,driver_name,constructor_id,constructor_name,grid,position,position_text,points,laps,status,finish_time,fastest_lap_rank,fastest_lap_time,position_num
0,2018,1,Australian Grand Prix,2018-03-25,vettel,VET,5,Sebastian Vettel,ferrari,Ferrari,3,1,1,25,58,Finished,1:29:33.283,4,1:26.469,1
1,2018,1,Australian Grand Prix,2018-03-25,hamilton,HAM,44,Lewis Hamilton,mercedes,Mercedes,1,2,2,18,58,Finished,+5.036,3,1:26.444,2
2,2018,1,Australian Grand Prix,2018-03-25,raikkonen,RAI,7,Kimi Räikkönen,ferrari,Ferrari,2,3,3,15,58,Finished,+6.309,2,1:26.373,3
3,2018,1,Australian Grand Prix,2018-03-25,ricciardo,RIC,3,Daniel Ricciardo,red_bull,Red Bull,8,4,4,12,58,Finished,+7.069,1,1:25.945,4
4,2018,1,Australian Grand Prix,2018-03-25,alonso,ALO,14,Fernando Alonso,mclaren,McLaren,10,5,5,10,58,Finished,+27.886,7,1:26.978,5


In [14]:

race_summary = (
    results_df
    .groupby(["season", "round", "race_name"], as_index=False)
    .agg(
        drivers_recorded=("driver_id", "nunique"),
        winners=("position_num", lambda s: (s == 1).sum()),
        race_date=("race_date", "first"),
    )
)

print("Races with exactly one winner:",
      (race_summary["winners"] == 1).sum())

print("Races with no recorded winner:",
      (race_summary["winners"] == 0).sum())

print("Races with multiple recorded winners:",
      (race_summary["winners"] > 1).sum())

print("\nMost recent races:")
display(race_summary.tail(10))

Races with exactly one winner: 6
Races with no recorded winner: 0
Races with multiple recorded winners: 0

Most recent races:


,season,round,race_name,drivers_recorded,winners,race_date
0,2018,1,Australian Grand Prix,20,1,2018-03-25
1,2018,2,Bahrain Grand Prix,20,1,2018-04-08
2,2018,3,Chinese Grand Prix,20,1,2018-04-15
3,2018,4,Azerbaijan Grand Prix,20,1,2018-04-29
4,2018,5,Spanish Grand Prix,20,1,2018-05-13
5,2018,6,Monaco Grand Prix,20,1,2018-05-27


In [15]:

results_path = RAW_DIR / "race_results.csv"

results_df.drop(
    columns=["position_num"],
    errors="ignore"
).to_csv(results_path, index=False)

print("Saved:", results_path.resolve())

Saved: C:\Users\saksham\Desktop\Project\f1-prediction-platform\data\raw\race_results.csv


In [16]:

race_summary = (
    results_df
    .groupby(["season", "round", "race_name"], as_index=False)
    .agg(
        drivers_recorded=("driver_id", "nunique"),
        winners=("position_num", lambda s: (s == 1).sum()),
        race_date=("race_date", "first"),
    )
)

print("Total races:", len(race_summary))
print("Exactly one winner:",
      (race_summary["winners"] == 1).sum())
print("No recorded winner:",
      (race_summary["winners"] == 0).sum())
print("Multiple recorded winners:",
      (race_summary["winners"] > 1).sum())

display(
    race_summary[race_summary["winners"] != 1]
)

Total races: 6
Exactly one winner: 6
No recorded winner: 0
Multiple recorded winners: 0


,season,round,race_name,drivers_recorded,winners,race_date


In [17]:

results_path = RAW_DIR / "race_results.csv"

results_df.drop(
    columns=["position_num"],
    errors="ignore"
).to_csv(results_path, index=False)

print("Saved to:", results_path.resolve())

Saved to: C:\Users\saksham\Desktop\Project\f1-prediction-platform\data\raw\race_results.csv


In [18]:

qualifying_records = []

for season in SEASONS:
    season_count = 0

    season_races = (
        calendar_df[calendar_df["season"] == season]
        .sort_values("round")
    )

    for _, race_row in season_races.iterrows():
        round_number = int(race_row["round"])
        url = f"{BASE_URL}/{season}/{round_number}/qualifying.json"

        payload = get_json_with_retry(
            url,
            params={"limit": 100, "format": "json"}
        )

        races = payload["MRData"]["RaceTable"].get("Races", [])

        for race in races:
            for result in race.get("QualifyingResults", []):
                driver = result.get("Driver", {})
                constructor = result.get("Constructor", {})

                qualifying_records.append({
                    "season": int(race["season"]),
                    "round": int(race["round"]),
                    "race_name": race["raceName"],
                    "race_date": race.get("date"),
                    "driver_id": driver.get("driverId"),
                    "driver_code": driver.get("code"),
                    "driver_name": " ".join(filter(None, [
                        driver.get("givenName"),
                        driver.get("familyName")
                    ])),
                    "constructor_id": constructor.get("constructorId"),
                    "constructor_name": constructor.get("name"),
                    "qualifying_position": result.get("position"),
                    "q1_time": result.get("Q1"),
                    "q2_time": result.get("Q2"),
                    "q3_time": result.get("Q3"),
                })

                season_count += 1

        time.sleep(1.0)

    print(f"{season}: {season_count} qualifying records")

2018: 420 qualifying records
2019: 418 qualifying records
2020: 340 qualifying records
2021: 439 qualifying records
2022: 440 qualifying records
2023: 440 qualifying records
2024: 479 qualifying records
2025: 479 qualifying records
2026: 369 qualifying records


In [19]:

qualifying_df = pd.DataFrame(qualifying_records)

if not qualifying_df.empty:
    qualifying_df = (
        qualifying_df
        .drop_duplicates(subset=["season", "round", "driver_id"])
        .sort_values(["season", "round"])
        .reset_index(drop=True)
    )

    qualifying_path = RAW_DIR / "qualifying_results_partial.csv"
    qualifying_df.to_csv(qualifying_path, index=False)

    print("Saved partial qualifying data:", qualifying_path.resolve())
    print("Rows saved:", len(qualifying_df))
    print("Seasons saved:", sorted(qualifying_df["season"].unique()))
else:
    print("No qualifying records are currently in memory.")

Saved partial qualifying data: C:\Users\saksham\Desktop\Project\f1-prediction-platform\data\raw\qualifying_results_partial.csv
Rows saved: 3824
Seasons saved: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]


In [ ]:

from pathlib import Path
import pandas as pd

cwd = Path.cwd()
PROJECT_DIR = cwd.parent if cwd.name == "notebooks" else cwd
RAW_DIR = PROJECT_DIR / "data" / "raw"

print("Raw data folder:", RAW_DIR.resolve())

for filename in [
    "qualifying_results.csv",
    "qualifying_results_partial.csv",
    "race_results.csv",
    "race_calendar.csv",
]:
    path = RAW_DIR / filename
    if path.exists():
        print(f"FOUND: {filename} ({path.stat().st_size:,} bytes)")
    else:
        print(f"NOT FOUND: {filename}")

Raw data folder: C:\Users\saksham\Desktop\Project\f1-prediction-platform\data\raw
NOT FOUND: qualifying_results.csv
NOT FOUND: qualifying_results_partial.csv
FOUND: race_results.csv (483,993 bytes)
FOUND: race_calendar.csv (33,453 bytes)


In [ ]:

import time
import requests
import pandas as pd
from pathlib import Path

BASE_URL = "https://api.jolpi.ca/ergast/f1"

def get_json_with_retry(url, params=None, max_attempts=5):
    for attempt in range(max_attempts):
        try:
            response = requests.get(url, params=params, timeout=45)

            if response.status_code == 429 or response.status_code in {
                500, 502, 503, 504
            }:
                if attempt == max_attempts - 1:
                    response.raise_for_status()

                wait = min(15 * (2 ** attempt), 120)
                print(f"HTTP {response.status_code}; waiting {wait}s...")
                time.sleep(wait)
                continue

            response.raise_for_status()
            return response.json()

        except requests.RequestException:
            if attempt == max_attempts - 1:
                raise
            time.sleep(min(15 * (2 ** attempt), 120))

    raise RuntimeError(f"Failed to retrieve {url}")


In [20]:

season = 2025
season_records = []

calendar_path = RAW_DIR / "race_calendar.csv"
calendar_df = pd.read_csv(calendar_path)

season_races = (
    calendar_df[calendar_df["season"] == season]
    .sort_values("round")
)

for _, race_row in season_races.iterrows():
    round_number = int(race_row["round"])
    url = f"{BASE_URL}/{season}/{round_number}/qualifying.json"

    payload = get_json_with_retry(
        url,
        params={"limit": 100, "format": "json"}
    )

    races = payload["MRData"]["RaceTable"].get("Races", [])

    for race in races:
        for result in race.get("QualifyingResults", []):
            driver = result.get("Driver", {})
            constructor = result.get("Constructor", {})

            season_records.append({
                "season": int(race["season"]),
                "round": int(race["round"]),
                "race_name": race["raceName"],
                "race_date": race.get("date"),
                "driver_id": driver.get("driverId"),
                "driver_code": driver.get("code"),
                "driver_name": " ".join(filter(None, [
                    driver.get("givenName"),
                    driver.get("familyName")
                ])),
                "constructor_id": constructor.get("constructorId"),
                "constructor_name": constructor.get("name"),
                "qualifying_position": result.get("position"),
                "q1_time": result.get("Q1"),
                "q2_time": result.get("Q2"),
                "q3_time": result.get("Q3"),
            })

    print(f"Completed {season} round {round_number}")
    time.sleep(5)

season_df = pd.DataFrame(season_records)

if not season_df.empty:
    season_df = (
        season_df
        .drop_duplicates(subset=["season", "round", "driver_id"])
        .sort_values(["season", "round", "qualifying_position"])
        .reset_index(drop=True)
    )

season_path = RAW_DIR / f"qualifying_{season}.csv"
season_df.to_csv(season_path, index=False)

print("\nSaved:", season_path.resolve())
print("Shape:", season_df.shape)
print("Races with qualifying data:",
      season_df["round"].nunique() if not season_df.empty else 0)

Completed 2025 round 1
Completed 2025 round 2
Completed 2025 round 3
Completed 2025 round 4
Completed 2025 round 5
Completed 2025 round 6
Completed 2025 round 7
Completed 2025 round 8
Completed 2025 round 9
Completed 2025 round 10
Completed 2025 round 11
Completed 2025 round 12
Completed 2025 round 13
Completed 2025 round 14
Completed 2025 round 15
Completed 2025 round 16
Completed 2025 round 17
Completed 2025 round 18
Completed 2025 round 19
Completed 2025 round 20
Completed 2025 round 21
Completed 2025 round 22
Completed 2025 round 23
Completed 2025 round 24

Saved: C:\Users\saksham\Desktop\Project\f1-prediction-platform\data\raw\qualifying_2025.csv
Shape: (479, 13)
Races with qualifying data: 24


In [ ]:

from pathlib import Path
import pandas as pd

# Recover the records already collected in this kernel.
if "season_records" not in globals():
    print("season_records is missing from memory.")
else:
    partial_2025 = pd.DataFrame(season_records)

    if not partial_2025.empty:
        partial_2025 = (
            partial_2025
            .drop_duplicates(
                subset=["season", "round", "driver_id"]
            )
            .sort_values(
                ["season", "round", "qualifying_position"]
            )
            .reset_index(drop=True)
        )

        save_path = RAW_DIR / "qualifying_2025_partial.csv"
        partial_2025.to_csv(save_path, index=False)

        print("Saved:", save_path.resolve())
        print("Rows saved:", len(partial_2025))
        print(
            "Rounds saved:",
            sorted(partial_2025["round"].unique())
        )
    else:
        print("No qualifying records were collected.")

Saved: C:\Users\saksham\Desktop\Project\f1-prediction-platform\data\raw\qualifying_2025_partial.csv
Rows saved: 260
Rounds saved: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13)]


In [ ]:
check_df = pd.read_csv(RAW_DIR / "qualifying_2025_partial.csv")

print("Shape:", check_df.shape)
display(
    check_df.groupby("round").agg(
        drivers=("driver_id", "nunique"),
        race=("race_name", "first"),
    )
)

Shape: (260, 13)


,drivers,race
round,,
1,20,Australian Grand Prix
2,20,Chinese Grand Prix
3,20,Japanese Grand Prix
4,20,Bahrain Grand Prix
5,20,Saudi Arabian Grand Prix
6,20,Miami Grand Prix
7,20,Emilia Romagna Grand Prix
8,20,Monaco Grand Prix
9,20,Spanish Grand Prix


In [ ]:

import time
import pandas as pd

season = 2025
save_path = RAW_DIR / "qualifying_2025_partial.csv"

# Load existing progress
if save_path.exists():
    existing_df = pd.read_csv(save_path)
else:
    existing_df = pd.DataFrame()

completed_rounds = (
    set(existing_df["round"].dropna().astype(int))
    if not existing_df.empty
    else set()
)

print("Already saved rounds:", sorted(completed_rounds))

calendar_df = pd.read_csv(RAW_DIR / "race_calendar.csv")
season_races = (
    calendar_df[calendar_df["season"] == season]
    .sort_values("round")
)

for _, race_row in season_races.iterrows():
    round_number = int(race_row["round"])

    if round_number in completed_rounds:
        continue

    url = f"{BASE_URL}/{season}/{round_number}/qualifying.json"

    # Use a long cooldown if the API rate-limits us.
    for attempt in range(5):
        try:
            response = requests.get(
                url,
                params={"limit": 100, "format": "json"},
                timeout=45,
            )

            if response.status_code == 429:
                wait_seconds = 120 * (attempt + 1)
                print(
                    f"Round {round_number}: HTTP 429. "
                    f"Waiting {wait_seconds}s."
                )
                if attempt == 4:
                    response.raise_for_status()
                time.sleep(wait_seconds)
                continue

            response.raise_for_status()
            payload = response.json()
            break

        except requests.RequestException:
            if attempt == 4:
                raise
            wait_seconds = 60 * (attempt + 1)
            print(f"Temporary error. Waiting {wait_seconds}s.")
            time.sleep(wait_seconds)

    else:
        raise RuntimeError(f"Could not retrieve round {round_number}")

    new_records = []

    races = payload["MRData"]["RaceTable"].get("Races", [])

    for race in races:
        for result in race.get("QualifyingResults", []):
            driver = result.get("Driver", {})
            constructor = result.get("Constructor", {})

            new_records.append({
                "season": int(race["season"]),
                "round": int(race["round"]),
                "race_name": race["raceName"],
                "race_date": race.get("date"),
                "driver_id": driver.get("driverId"),
                "driver_code": driver.get("code"),
                "driver_name": " ".join(filter(None, [
                    driver.get("givenName"),
                    driver.get("familyName")
                ])),
                "constructor_id": constructor.get("constructorId"),
                "constructor_name": constructor.get("name"),
                "qualifying_position": result.get("position"),
                "q1_time": result.get("Q1"),
                "q2_time": result.get("Q2"),
                "q3_time": result.get("Q3"),
            })

    if new_records:
        round_df = pd.DataFrame(new_records)

        existing_df = pd.concat(
            [existing_df, round_df],
            ignore_index=True
        )

        existing_df = (
            existing_df
            .drop_duplicates(
                subset=["season", "round", "driver_id"],
                keep="last"
            )
            .sort_values(["season", "round", "qualifying_position"])
            .reset_index(drop=True)
        )

        # Save immediately after each successful round.
        existing_df.to_csv(save_path, index=False)
        completed_rounds.add(round_number)

        print(
            f"Saved round {round_number}: "
            f"{len(new_records)} records; "
            f"{len(completed_rounds)} rounds saved"
        )
    else:
        print(
            f"Round {round_number} returned no qualifying records. "
            "It remains uncollected."
        )

    time.sleep(10)

print("\nCollection pass finished.")
print("Saved file:", save_path.resolve())
print("Total rows:", len(existing_df))
print("Rounds saved:", sorted(completed_rounds))

Already saved rounds: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]
Saved round 14: 20 records; 14 rounds saved
Saved round 15: 20 records; 15 rounds saved
Saved round 16: 20 records; 16 rounds saved
Saved round 17: 20 records; 17 rounds saved
Saved round 18: 20 records; 18 rounds saved
Saved round 19: 20 records; 19 rounds saved
Saved round 20: 20 records; 20 rounds saved
Saved round 21: 19 records; 21 rounds saved
Saved round 22: 20 records; 22 rounds saved
Saved round 23: 20 records; 23 rounds saved
Saved round 24: 20 records; 24 rounds saved

Collection pass finished.
Saved file: C:\Users\saksham\Desktop\Project\f1-prediction-platform\data\raw\qualifying_2025_partial.csv
Total rows: 479
Rounds saved: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]


In [ ]:
check_df = pd.read_csv(RAW_DIR / "qualifying_2025_partial.csv")

print("Shape:", check_df.shape)
print("Rounds collected:", sorted(check_df["round"].unique()))
print("Missing rounds:", sorted(set(range(1, 25)) - set(
    check_df["round"].dropna().astype(int)
)))

Shape: (479, 13)
Rounds collected: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24)]
Missing rounds: []


In [ ]:

from pathlib import Path

print("Qualifying files currently saved:")

for path in sorted(RAW_DIR.glob("qualifying*.csv")):
    df = pd.read_csv(path)
    seasons = (
        sorted(df["season"].dropna().astype(int).unique())
        if "season" in df.columns
        else []
    )
    print(
        f"{path.name}: {len(df)} rows | seasons: {seasons}"
    )

Qualifying files currently saved:
qualifying_2025_partial.csv: 479 rows | seasons: [np.int64(2025)]


In [ ]:

import time
import requests
import pandas as pd

season = 2024
save_path = RAW_DIR / f"qualifying_{season}_partial.csv"

# Load previously saved progress for this season, if any.
if save_path.exists():
    season_df = pd.read_csv(save_path)
else:
    season_df = pd.DataFrame()

completed_rounds = (
    set(season_df["round"].dropna().astype(int))
    if not season_df.empty
    else set()
)

calendar_df = pd.read_csv(RAW_DIR / "race_calendar.csv")
season_races = (
    calendar_df[calendar_df["season"] == season]
    .sort_values("round")
)

print("Season:", season)
print("Rounds already saved:", sorted(completed_rounds))

for _, race_row in season_races.iterrows():
    round_number = int(race_row["round"])

    if round_number in completed_rounds:
        continue

    url = f"{BASE_URL}/{season}/{round_number}/qualifying.json"
    payload = None

    for attempt in range(5):
        try:
            response = requests.get(
                url,
                params={"limit": 100, "format": "json"},
                timeout=45,
            )

            if response.status_code == 429:
                wait_seconds = 120 * (attempt + 1)
                print(
                    f"Round {round_number}: HTTP 429; "
                    f"waiting {wait_seconds}s."
                )

                if attempt == 4:
                    response.raise_for_status()

                time.sleep(wait_seconds)
                continue

            response.raise_for_status()
            payload = response.json()
            break

        except requests.RequestException:
            if attempt == 4:
                raise

            wait_seconds = 60 * (attempt + 1)
            print(f"Temporary error; waiting {wait_seconds}s.")
            time.sleep(wait_seconds)

    if payload is None:
        print(f"Stopping at round {round_number}; no response received.")
        break

    new_records = []

    for race in payload["MRData"]["RaceTable"].get("Races", []):
        for result in race.get("QualifyingResults", []):
            driver = result.get("Driver", {})
            constructor = result.get("Constructor", {})

            new_records.append({
                "season": int(race["season"]),
                "round": int(race["round"]),
                "race_name": race["raceName"],
                "race_date": race.get("date"),
                "driver_id": driver.get("driverId"),
                "driver_code": driver.get("code"),
                "driver_name": " ".join(filter(None, [
                    driver.get("givenName"),
                    driver.get("familyName")
                ])),
                "constructor_id": constructor.get("constructorId"),
                "constructor_name": constructor.get("name"),
                "qualifying_position": result.get("position"),
                "q1_time": result.get("Q1"),
                "q2_time": result.get("Q2"),
                "q3_time": result.get("Q3"),
            })

    if new_records:
        season_df = pd.concat(
            [season_df, pd.DataFrame(new_records)],
            ignore_index=True
        )

        season_df = (
            season_df
            .drop_duplicates(
                subset=["season", "round", "driver_id"],
                keep="last"
            )
            .sort_values(["season", "round", "qualifying_position"])
            .reset_index(drop=True)
        )

        # Checkpoint after every successful round.
        season_df.to_csv(save_path, index=False)
        completed_rounds.add(round_number)

        print(
            f"Saved round {round_number}: {len(new_records)} records; "
            f"{len(completed_rounds)} rounds saved."
        )
    else:
        print(
            f"Round {round_number} returned no qualifying records; "
            "it will remain missing."
        )

    time.sleep(10)

print("\nCollection pass complete.")
print("File:", save_path.resolve())
print("Rows:", len(season_df))
print("Rounds saved:", sorted(completed_rounds))

Season: 2024
Rounds already saved: []
Saved round 1: 20 records; 1 rounds saved.
Saved round 2: 20 records; 2 rounds saved.
Saved round 3: 19 records; 3 rounds saved.
Saved round 4: 20 records; 4 rounds saved.
Saved round 5: 20 records; 5 rounds saved.
Saved round 6: 20 records; 6 rounds saved.
Saved round 7: 20 records; 7 rounds saved.
Saved round 8: 20 records; 8 rounds saved.
Saved round 9: 20 records; 9 rounds saved.
Saved round 10: 20 records; 10 rounds saved.
Saved round 11: 20 records; 11 rounds saved.
Saved round 12: 20 records; 12 rounds saved.
Saved round 13: 20 records; 13 rounds saved.
Saved round 14: 20 records; 14 rounds saved.
Saved round 15: 20 records; 15 rounds saved.
Saved round 16: 20 records; 16 rounds saved.
Saved round 17: 20 records; 17 rounds saved.
Saved round 18: 20 records; 18 rounds saved.
Saved round 19: 20 records; 19 rounds saved.
Saved round 20: 20 records; 20 rounds saved.
Saved round 21: 20 records; 21 rounds saved.
Saved round 22: 20 records; 22 roun

In [ ]:
season = 2023
save_path = RAW_DIR / f"qualifying_{season}_partial.csv"

In [21]:
season_df = pd.read_csv(
    RAW_DIR / "qualifying_2023_partial.csv"
)

print("Shape:", season_df.shape)
print("Rounds:", sorted(season_df["round"].unique()))
print("Unique drivers:", season_df["driver_id"].nunique())

Shape: (440, 13)
Rounds: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22)]
Unique drivers: 22


In [ ]:

from pathlib import Path
import pandas as pd

print("Raw data folder:", RAW_DIR.resolve())

for path in sorted(RAW_DIR.glob("qualifying*.csv")):
    df = pd.read_csv(path)
    print(
        f"{path.name}: {len(df)} rows, "
        f"{df['round'].nunique()} rounds"
    )

Raw data folder: C:\Users\saksham\Desktop\Project\f1-prediction-platform\data\raw
qualifying_2024_partial.csv: 479 rows, 24 rounds
qualifying_2025_partial.csv: 479 rows, 24 rounds


In [ ]:
season = 2023
save_path = RAW_DIR / f"qualifying_{season}_partial.csv"

In [22]:
season_df = pd.read_csv(RAW_DIR / "qualifying_2023_partial.csv")

print("Shape:", season_df.shape)
print("Rounds collected:", sorted(season_df["round"].unique()))

Shape: (440, 13)
Rounds collected: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22)]


In [ ]:
from pathlib import Path

print("Current working directory:", Path.cwd())
print("RAW_DIR:", RAW_DIR.resolve())
print("Raw folder exists:", RAW_DIR.exists())

print("\nQualifying files found:")
files = list(RAW_DIR.glob("qualifying*.csv"))

if files:
    for file in sorted(files):
        print(f" - {file.name} ({file.stat().st_size:,} bytes)")
else:
    print("No qualifying CSV files found in RAW_DIR.")

Current working directory: c:\Users\saksham\Desktop\Project\f1-prediction-platform\notebooks
RAW_DIR: C:\Users\saksham\Desktop\Project\f1-prediction-platform\data\raw
Raw folder exists: True

Qualifying files found:
 - qualifying_2024_partial.csv (51,981 bytes)
 - qualifying_2025_partial.csv (51,694 bytes)


In [ ]:

import time
import requests
import pandas as pd
from pathlib import Path

# Settings
season = 2023
save_path = RAW_DIR / f"qualifying_{season}_partial.csv"
calendar_path = RAW_DIR / "race_calendar.csv"
API_LIMIT = 100

# Load the race calendar
calendar = pd.read_csv(calendar_path)
season_calendar = calendar[calendar["season"] == season].copy()
season_calendar = season_calendar.sort_values("round")

if season_calendar.empty:
    raise ValueError(
        f"No races found for {season} in {calendar_path}. "
        "Check the season column and calendar file."
    )

# Resume from the existing checkpoint, if present
if save_path.exists():
    qualifying_df = pd.read_csv(save_path)
    completed_rounds = set(
        qualifying_df["round"].dropna().astype(int).unique()
    )
else:
    qualifying_df = pd.DataFrame()
    completed_rounds = set()

print(f"Season: {season}")
print(f"Rounds in calendar: {len(season_calendar)}")
print(f"Rounds already saved: {sorted(completed_rounds)}")

session = requests.Session()
stop_collection = False

for _, calendar_race in season_calendar.iterrows():
    round_number = int(calendar_race["round"])

    if round_number in completed_rounds:
        print(f"Round {round_number}: already saved, skipping.")
        continue

    url = f"{BASE_URL}/{season}/{round_number}/qualifying.json"
    print(f"\nFetching {season}, round {round_number}...")

    response = None

    for attempt in range(1, 4):
        try:
            response = session.get(
                url,
                params={"limit": API_LIMIT, "format": "json"},
                timeout=45
            )

            if response.status_code == 429:
                print(
                    "API rate limit reached (HTTP 429). "
                    "Stopping safely; saved rounds will be kept."
                )
                stop_collection = True
                break

            response.raise_for_status()
            break

        except requests.RequestException as exc:
            print(f"Attempt {attempt}/3 failed: {exc}")
            if attempt < 3:
                time.sleep(15 * attempt)
            else:
                print("Could not fetch this round. Stopping safely.")
                stop_collection = True

    if stop_collection:
        break

    if response is None:
        break

    data = response.json()["MRData"]["RaceTable"].get("Races", [])

    if not data:
        print(f"No qualifying data returned for round {round_number}.")
        time.sleep(10)
        continue

    race = data[0]
    rows = []

    for result in race.get("QualifyingResults", []):
        driver = result.get("Driver", {})
        constructor = result.get("Constructor", {})

        rows.append({
            "season": int(race["season"]),
            "round": int(race["round"]),
            "race_name": race.get("raceName"),
            "race_date": race.get("date"),
            "driver_id": driver.get("driverId"),
            "driver_code": driver.get("code"),
            "driver_name": " ".join(
                filter(None, [
                    driver.get("givenName"),
                    driver.get("familyName")
                ])
            ),
            "constructor_id": constructor.get("constructorId"),
            "constructor_name": constructor.get("name"),
            "qualifying_position": result.get("position"),
            "q1_time": result.get("Q1"),
            "q2_time": result.get("Q2"),
            "q3_time": result.get("Q3"),
        })

    if rows:
        new_df = pd.DataFrame(rows)

        if qualifying_df.empty:
            qualifying_df = new_df
        else:
            qualifying_df = pd.concat(
                [qualifying_df, new_df], ignore_index=True
            )

        qualifying_df = qualifying_df.drop_duplicates(
            subset=["season", "round", "driver_id"],
            keep="last"
        )
        qualifying_df = qualifying_df.sort_values(
            ["season", "round", "qualifying_position"]
        )

        # Checkpoint after each successful round
        qualifying_df.to_csv(save_path, index=False)
        completed_rounds.add(round_number)

        print(
            f"Saved round {round_number}: {len(rows)} drivers. "
            f"Total saved: {len(qualifying_df)} rows."
        )
    else:
        print(f"No qualifying results found for round {round_number}.")

    time.sleep(10)

print("\nCollection cell finished.")
print("File:", save_path)
print("File exists:", save_path.exists())

if save_path.exists():
    check_df = pd.read_csv(save_path)
    print("Rows saved:", len(check_df))
    print("Rounds saved:", sorted(check_df["round"].unique()))

Season: 2023
Rounds in calendar: 22
Rounds already saved: []

Fetching 2023, round 1...
Saved round 1: 20 drivers. Total saved: 20 rows.

Fetching 2023, round 2...
Saved round 2: 20 drivers. Total saved: 40 rows.

Fetching 2023, round 3...
Saved round 3: 20 drivers. Total saved: 60 rows.

Fetching 2023, round 4...
Saved round 4: 20 drivers. Total saved: 80 rows.

Fetching 2023, round 5...
Saved round 5: 20 drivers. Total saved: 100 rows.

Fetching 2023, round 6...
Saved round 6: 20 drivers. Total saved: 120 rows.

Fetching 2023, round 7...
Saved round 7: 20 drivers. Total saved: 140 rows.

Fetching 2023, round 8...
Saved round 8: 20 drivers. Total saved: 160 rows.

Fetching 2023, round 9...
Saved round 9: 20 drivers. Total saved: 180 rows.

Fetching 2023, round 10...
Saved round 10: 20 drivers. Total saved: 200 rows.

Fetching 2023, round 11...
Saved round 11: 20 drivers. Total saved: 220 rows.

Fetching 2023, round 12...
Saved round 12: 20 drivers. Total saved: 240 rows.

Fetching 202

In [ ]:

import pandas as pd

qualifying_2023 = pd.read_csv(RAW_DIR / "qualifying_2023_partial.csv")

print("Dataset shape:", qualifying_2023.shape)
print("Rounds collected:", sorted(qualifying_2023["round"].unique()))
print("Unique drivers:", qualifying_2023["driver_id"].nunique())
print("Missing values:")
display(qualifying_2023.isnull().sum().to_frame("missing_count"))

print("\nDrivers recorded per round:")
display(
    qualifying_2023.groupby(["round", "race_name"])
    .agg(
        drivers=("driver_id", "nunique"),
        duplicate_driver_rows=("driver_id", lambda s: s.duplicated().sum())
    )
    .reset_index()
)

Dataset shape: (440, 13)
Rounds collected: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22)]
Unique drivers: 22
Missing values:


,missing_count
season,0
round,0
race_name,0
race_date,0
driver_id,0
driver_code,0
driver_name,0
constructor_id,0
constructor_name,0
qualifying_position,0



Drivers recorded per round:


,round,race_name,drivers,duplicate_driver_rows
0,1,Bahrain Grand Prix,20,0
1,2,Saudi Arabian Grand Prix,20,0
2,3,Australian Grand Prix,20,0
3,4,Azerbaijan Grand Prix,20,0
4,5,Miami Grand Prix,20,0
5,6,Monaco Grand Prix,20,0
6,7,Spanish Grand Prix,20,0
7,8,Canadian Grand Prix,20,0
8,9,Austrian Grand Prix,20,0
9,10,British Grand Prix,20,0


In [35]:

import pandas as pd

seasons_to_combine = [2023, 2024, 2025]
qualifying_dfs = []

for season in seasons_to_combine:
    file_path = RAW_DIR / f"qualifying_{season}_partial.csv"

    if not file_path.exists():
        print(f"Missing file: {file_path.name}")
        continue

    df = pd.read_csv(file_path)
    qualifying_dfs.append(df)

    print(
        f"{season}: {len(df)} rows, "
        f"{df['round'].nunique()} rounds"
    )

if len(qualifying_dfs) == len(seasons_to_combine):
    qualifying_all = pd.concat(qualifying_dfs, ignore_index=True)

    qualifying_all = qualifying_all.drop_duplicates(
        subset=["season", "round", "driver_id"],
        keep="last"
    )

    qualifying_all = qualifying_all.sort_values(
        ["season", "round", "qualifying_position"]
    ).reset_index(drop=True)

    output_path = RAW_DIR / "qualifying_2023_2025.csv"
    qualifying_all.to_csv(output_path, index=False)

    print("\nCombined dataset saved:", output_path)
    print("Combined shape:", qualifying_all.shape)

    display(
        qualifying_all.groupby("season").agg(
            rows=("driver_id", "size"),
            races=("round", "nunique"),
            drivers=("driver_id", "nunique")
        ).reset_index()
    )
else:
    print("Not all files are available. Check the messages above.")

2023: 440 rows, 22 rounds
2024: 479 rows, 24 rounds
2025: 479 rows, 24 rounds

Combined dataset saved: c:\Users\saksham\Desktop\Project\f1-prediction-platform\data\raw\qualifying_2023_2025.csv
Combined shape: (1398, 13)


,season,rows,races,drivers
0,2023,440,22,22
1,2024,479,24,24
2,2025,479,24,21


In [36]:

race_counts = (
    qualifying_all.groupby(["season", "round", "race_name"])
    .agg(
        rows=("driver_id", "size"),
        unique_drivers=("driver_id", "nunique"),
        missing_positions=("qualifying_position", lambda s: s.isna().sum())
    )
    .reset_index()
)

print("Races with fewer than 20 records:")
display(race_counts[race_counts["rows"] < 20])

print("\nRaces with duplicate driver records:")
display(race_counts[race_counts["rows"] != race_counts["unique_drivers"]])

print("\nQualifying position data type:")
print(qualifying_all["qualifying_position"].dtype)

Races with fewer than 20 records:


,season,round,race_name,rows,unique_drivers,missing_positions
24,2024,3,Australian Grand Prix,19,19,0
66,2025,21,São Paulo Grand Prix,19,19,0



Races with duplicate driver records:


,season,round,race_name,rows,unique_drivers,missing_positions



Qualifying position data type:
int64


In [37]:

import requests
import pandas as pd

races_to_check = [
    (2024, 3, "Australian Grand Prix"),
    (2025, 21, "São Paulo Grand Prix"),
]

for season, round_number, race_name in races_to_check:
    url = f"{BASE_URL}/{season}/{round_number}/qualifying.json"

    response = requests.get(
        url,
        params={"limit": 100, "format": "json"},
        timeout=30
    )
    response.raise_for_status()

    races = response.json()["MRData"]["RaceTable"].get("Races", [])

    print(f"\n{season} — {race_name}")

    if not races:
        print("No race data returned by the API.")
        continue

    results = races[0].get("QualifyingResults", [])
    api_drivers = {
        result["Driver"]["driverId"]
        for result in results
    }

    saved_drivers = set(
        qualifying_all.loc[
            (qualifying_all["season"] == season)
            & (qualifying_all["round"] == round_number),
            "driver_id"
        ].dropna()
    )

    print("API records:", len(results))
    print("Saved records:", len(saved_drivers))
    print("Drivers in API but missing from saved file:",
          sorted(api_drivers - saved_drivers))
    print("Drivers in saved file but not in API:",
          sorted(saved_drivers - api_drivers))


2024 — Australian Grand Prix
API records: 19
Saved records: 19
Drivers in API but missing from saved file: []
Drivers in saved file but not in API: []

2025 — São Paulo Grand Prix
API records: 19
Saved records: 19
Drivers in API but missing from saved file: []
Drivers in saved file but not in API: []


In [31]:

from pathlib import Path
import pandas as pd

results_path = RAW_DIR / "race_results.csv"

if not results_path.exists():
    print("Race results file not found:", results_path)
else:
    results_df = pd.read_csv(results_path)

    print("Race results shape:", results_df.shape)
    print("\nColumns:")
    print(results_df.columns.tolist())

    print("\nSeason coverage:")
    display(
        results_df.groupby("season").agg(
            races=("round", "nunique"),
            result_rows=("driver_id", "size"),
            unique_drivers=("driver_id", "nunique")
        ).reset_index()
    )

    print("\nSample records:")
    display(results_df.head())

Race results shape: (120, 19)

Columns:
['season', 'round', 'race_name', 'race_date', 'driver_id', 'driver_code', 'driver_number', 'driver_name', 'constructor_id', 'constructor_name', 'grid', 'position', 'position_text', 'points', 'laps', 'status', 'finish_time', 'fastest_lap_rank', 'fastest_lap_time']

Season coverage:


,season,races,result_rows,unique_drivers
0,2018,6,120,20



Sample records:


,season,round,race_name,race_date,driver_id,driver_code,driver_number,driver_name,constructor_id,constructor_name,grid,position,position_text,points,laps,status,finish_time,fastest_lap_rank,fastest_lap_time
0,2018,1,Australian Grand Prix,2018-03-25,vettel,VET,5,Sebastian Vettel,ferrari,Ferrari,3,1,1,25,58,Finished,1:29:33.283,4.0,1:26.469
1,2018,1,Australian Grand Prix,2018-03-25,hamilton,HAM,44,Lewis Hamilton,mercedes,Mercedes,1,2,2,18,58,Finished,+5.036,3.0,1:26.444
2,2018,1,Australian Grand Prix,2018-03-25,raikkonen,RAI,7,Kimi Räikkönen,ferrari,Ferrari,2,3,3,15,58,Finished,+6.309,2.0,1:26.373
3,2018,1,Australian Grand Prix,2018-03-25,ricciardo,RIC,3,Daniel Ricciardo,red_bull,Red Bull,8,4,4,12,58,Finished,+7.069,1.0,1:25.945
4,2018,1,Australian Grand Prix,2018-03-25,alonso,ALO,14,Fernando Alonso,mclaren,McLaren,10,5,5,10,58,Finished,+27.886,7.0,1:26.978


In [39]:

import pandas as pd

# Work with the seasons for which we have qualifying data
model_seasons = [2023, 2024, 2025]

results_subset = results_df[
    results_df["season"].isin(model_seasons)
].copy()

qualifying_subset = qualifying_all[
    qualifying_all["season"].isin(model_seasons)
].copy()

# Convert positions to numeric values
results_subset["position_num"] = pd.to_numeric(
    results_subset["position"], errors="coerce"
)

qualifying_subset["qualifying_position"] = pd.to_numeric(
    qualifying_subset["qualifying_position"], errors="coerce"
)

# Create the target: 1 if the driver finished in P1, otherwise 0
results_subset["won_race"] = (
    results_subset["position_num"] == 1
).astype(int)

# Keep only columns needed for this baseline
results_for_merge = results_subset[
    [
        "season", "round", "driver_id", "driver_name",
        "constructor_id", "constructor_name",
        "grid", "position_num", "won_race"
    ]
]

qualifying_for_merge = qualifying_subset[
    [
        "season", "round", "driver_id",
        "qualifying_position"
    ]
]

# Join on season, round, and driver
model_df = results_for_merge.merge(
    qualifying_for_merge,
    on=["season", "round", "driver_id"],
    how="left",
    validate="one_to_one"
)

# Save the baseline dataset
output_path = RAW_DIR / "baseline_model_dataset.csv"
model_df.to_csv(output_path, index=False)

print("Dataset saved to:", output_path)
print("Shape:", model_df.shape)
print("\nQualifying position coverage:")
print(model_df["qualifying_position"].notna().mean().round(4))

print("\nMissing qualifying positions by season:")
display(
    model_df.groupby("season")["qualifying_position"]
    .apply(lambda s: s.isna().sum())
    .reset_index(name="missing_qualifying_positions")
)

print("\nTarget distribution:")
display(model_df["won_race"].value_counts().rename_axis("won_race").to_frame("driver_races"))

print("\nSample:")
display(model_df.head(10))

Dataset saved to: c:\Users\saksham\Desktop\Project\f1-prediction-platform\data\raw\baseline_model_dataset.csv
Shape: (0, 10)

Qualifying position coverage:


AttributeError: 'float' object has no attribute 'round'

In [29]:

# 1. Inspect the record with a missing qualifying position
print("Records missing qualifying position:")
display(
    model_df[model_df["qualifying_position"].isna()][
        [
            "season", "round", "driver_name",
            "constructor_name", "grid", "position_num",
            "won_race"
        ]
    ]
)

# 2. Check that each race has exactly one winner
winner_check = (
    model_df.groupby(["season", "round"])
    .agg(
        driver_records=("driver_id", "size"),
        winners=("won_race", "sum")
    )
    .reset_index()
)

print("\nRace winner validation:")
print("Total races:", len(winner_check))
print("Races with exactly one winner:",
      (winner_check["winners"] == 1).sum())
print("Races with zero winners:",
      (winner_check["winners"] == 0).sum())
print("Races with multiple winners:",
      (winner_check["winners"] > 1).sum())

display(winner_check[winner_check["winners"] != 1])

Records missing qualifying position:


NameError: name 'model_df' is not defined

In [28]:

import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

# Copy the data so the original remains unchanged
baseline_df = model_df.copy()

# Use grid as a fallback only when qualifying position is missing
baseline_df["qualifying_position_feature"] = (
    baseline_df["qualifying_position"]
    .fillna(baseline_df["grid"])
)

# Features known before the race
feature_columns = ["qualifying_position_feature"]

# Chronological split: train on 2023-2024, test on 2025
train_df = baseline_df[baseline_df["season"].isin([2023, 2024])].copy()
test_df = baseline_df[baseline_df["season"] == 2025].copy()

X_train = train_df[feature_columns]
y_train = train_df["won_race"]

X_test = test_df[feature_columns]
y_test = test_df["won_race"]

# Preprocessing and model
baseline_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

baseline_model.fit(X_train, y_train)

# Predictions
y_pred = baseline_model.predict(X_test)
y_prob = baseline_model.predict_proba(X_test)[:, 1]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training seasons: 2023-2024")
print("Testing season: 2025")

print("\nEvaluation metrics")
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Winner precision:", round(precision_score(y_test, y_pred, zero_division=0), 4))
print("Winner recall:", round(recall_score(y_test, y_pred, zero_division=0), 4))
print("Winner F1:", round(f1_score(y_test, y_pred, zero_division=0), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))

print("\nClassification report:")
print(classification_report(
    y_test, y_pred,
    target_names=["Did not win", "Won"],
    zero_division=0
))

NameError: name 'model_df' is not defined

In [27]:

# Rebuild race-level evaluation using the original dataset
race_eval = model_df.loc[
    model_df["season"] == 2025,
    [
        "season", "round", "race_name", "driver_id",
        "driver_name", "constructor_name",
        "qualifying_position", "won_race"
    ]
].copy()

# Attach predictions in the same row order used for X_test
race_eval["predicted_win_probability"] = y_prob

# Rank drivers within each race (rank 1 = highest predicted probability)
race_eval["predicted_rank"] = (
    race_eval.groupby(["season", "round"])["predicted_win_probability"]
    .rank(method="first", ascending=False)
    .astype(int)
)

# Evaluate the rank of the actual winner in each race
winner_ranks = race_eval[race_eval["won_race"] == 1].copy()

total_races = len(winner_ranks)
top1_hits = (winner_ranks["predicted_rank"] == 1).sum()
top3_hits = (winner_ranks["predicted_rank"] <= 3).sum()

print("Race-level evaluation — 2025")
print("Races evaluated:", total_races)

if total_races:
    print(f"Top-1 winner accuracy: {top1_hits}/{total_races} = {top1_hits / total_races:.1%}")
    print(f"Actual winner ranked in top 3: {top3_hits}/{total_races} = {top3_hits / total_races:.1%}")
    print("Mean rank of actual winner:", round(winner_ranks["predicted_rank"].mean(), 2))

    print("\nActual winner's predicted rank:")
    display(
        winner_ranks[
            ["round", "race_name", "driver_name",
             "constructor_name", "predicted_rank"]
        ].sort_values("round").reset_index(drop=True)
    )

    print("\nTop predicted driver for each race:")
    display(
        race_eval[race_eval["predicted_rank"] == 1][
            ["round", "race_name", "driver_name",
             "constructor_name", "predicted_win_probability", "won_race"]
        ].sort_values("round").reset_index(drop=True)
    )
else:
    print("No race winners found. Check the won_race column.")

NameError: name 'model_df' is not defined

In [26]:

import pandas as pd

# Rebuild the 2025 test records from the original race results
race_eval = results_df.loc[
    results_df["season"] == 2025
].copy()

race_eval["position_num"] = pd.to_numeric(
    race_eval["position"], errors="coerce"
)
race_eval["won_race"] = (race_eval["position_num"] == 1).astype(int)

# Join qualifying position using driver and race identifiers
q2025 = qualifying_all.loc[
    qualifying_all["season"] == 2025,
    ["season", "round", "driver_id", "qualifying_position"]
].copy()

race_eval = race_eval.merge(
    q2025,
    on=["season", "round", "driver_id"],
    how="left",
    validate="one_to_one"
)

race_eval["qualifying_position_feature"] = (
    pd.to_numeric(race_eval["qualifying_position"], errors="coerce")
    .fillna(pd.to_numeric(race_eval["grid"], errors="coerce"))
)

# Predict directly from the rebuilt data
race_eval["predicted_win_probability"] = baseline_model.predict_proba(
    race_eval[["qualifying_position_feature"]]
)[:, 1]

# Rank drivers within each race
race_eval["predicted_rank"] = (
    race_eval.groupby(["season", "round"])["predicted_win_probability"]
    .rank(method="first", ascending=False)
    .astype(int)
)

winner_ranks = race_eval[race_eval["won_race"] == 1].copy()

total_races = len(winner_ranks)
top1_hits = (winner_ranks["predicted_rank"] == 1).sum()
top3_hits = (winner_ranks["predicted_rank"] <= 3).sum()

print("2025 race-level evaluation")
print("Races evaluated:", total_races)
print(f"Top-1 winner accuracy: {top1_hits}/{total_races} = {top1_hits / total_races:.1%}")
print(f"Actual winner in top 3: {top3_hits}/{total_races} = {top3_hits / total_races:.1%}")
print("Mean rank of actual winner:", round(winner_ranks["predicted_rank"].mean(), 2))

print("\nActual winner's predicted rank:")
display(
    winner_ranks[
        ["round", "race_name", "driver_name", "constructor_name", "predicted_rank"]
    ].sort_values("round").reset_index(drop=True)
)

print("\nTop predicted driver for each race:")
display(
    race_eval[race_eval["predicted_rank"] == 1][
        ["round", "race_name", "driver_name",
         "constructor_name", "predicted_win_probability", "won_race"]
    ].sort_values("round").reset_index(drop=True)
)

NameError: name 'qualifying_all' is not defined

In [ ]:

import pandas as pd

# Start from the complete historical race-results dataset
history = results_df.copy()

history["position_num"] = pd.to_numeric(
    history["position"], errors="coerce"
)
history["won_race"] = (
    history["position_num"] == 1
).astype(int)

# Sort chronologically before calculating historical features
history = history.sort_values(
    ["season", "round", "driver_id"]
).reset_index(drop=True)

# ----------------------------------------
# A. DRIVER HISTORICAL FEATURES
# ----------------------------------------

driver_groups = history.groupby("driver_id", sort=False)

# Average finish in races BEFORE the current race
history["driver_prior_mean_finish"] = (
    driver_groups["position_num"]
    .transform(lambda s: s.shift(1).expanding(min_periods=1).mean())
)

# Win rate BEFORE the current race
history["driver_prior_win_rate"] = (
    driver_groups["won_race"]
    .transform(lambda s: s.shift(1).expanding(min_periods=1).mean())
)

# Average finish across the previous five races
history["driver_recent5_mean_finish"] = (
    driver_groups["position_num"]
    .transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
)

# ----------------------------------------
# B. CONSTRUCTOR HISTORICAL FEATURES
# ----------------------------------------

# First aggregate each constructor's performance within each race.
# This prevents a teammate's result in the current race from
# leaking into the constructor's historical features.
constructor_races = (
    history.groupby(
        ["season", "round", "constructor_id"],
        as_index=False
    )
    .agg(
        constructor_race_mean_finish=("position_num", "mean"),
        constructor_race_won=("won_race", "max")
    )
    .sort_values(["season", "round", "constructor_id"])
    .reset_index(drop=True)
)

constructor_groups = constructor_races.groupby(
    "constructor_id", sort=False
)

constructor_races["constructor_prior_mean_finish"] = (
    constructor_groups["constructor_race_mean_finish"]
    .transform(lambda s: s.shift(1).expanding(min_periods=1).mean())
)

constructor_races["constructor_prior_win_rate"] = (
    constructor_groups["constructor_race_won"]
    .transform(lambda s: s.shift(1).expanding(min_periods=1).mean())
)

# Attach historical constructor features to each driver-race row
constructor_features = constructor_races[
    [
        "season", "round", "constructor_id",
        "constructor_prior_mean_finish",
        "constructor_prior_win_rate"
    ]
]

history = history.merge(
    constructor_features,
    on=["season", "round", "constructor_id"],
    how="left",
    validate="many_to_one"
)

# ----------------------------------------
# C. KEEP THE SEASONS WITH QUALIFYING DATA
# ----------------------------------------

qualifying_features = qualifying_all[
    ["season", "round", "driver_id", "qualifying_position"]
].copy()

model_features_df = history[
    history["season"].isin([2023, 2024, 2025])
].merge(
    qualifying_features,
    on=["season", "round", "driver_id"],
    how="left",
    validate="one_to_one"
)

# Use grid as a fallback only when qualifying position is missing
model_features_df["qualifying_position_feature"] = (
    pd.to_numeric(
        model_features_df["qualifying_position"], errors="coerce"
    ).fillna(
        pd.to_numeric(model_features_df["grid"], errors="coerce")
    )
)

# Save the engineered dataset
output_path = RAW_DIR.parent / "processed" / "historical_model_features.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
model_features_df.to_csv(output_path, index=False)

print("Engineered dataset saved:", output_path)
print("Dataset shape:", model_features_df.shape)

feature_columns = [
    "qualifying_position_feature",
    "driver_prior_mean_finish",
    "driver_prior_win_rate",
    "driver_recent5_mean_finish",
    "constructor_prior_mean_finish",
    "constructor_prior_win_rate"
]

print("\nFeature missing-value counts:")
display(
    model_features_df[feature_columns]
    .isna()
    .sum()
    .to_frame("missing_values")
)

print("\nSample historical features:")
display(
    model_features_df[
        [
            "season", "round", "driver_name", "constructor_name",
            *feature_columns, "won_race"
        ]
    ].head(10)
)

NameError: name 'results_df' is not defined

In [ ]:

import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from IPython.display import display

# 1. Define features
baseline_features = ["qualifying_position_feature"]

enhanced_features = [
    "qualifying_position_feature",
    "driver_prior_mean_finish",
    "driver_prior_win_rate",
    "driver_recent5_mean_finish",
    "constructor_prior_mean_finish",
    "constructor_prior_win_rate"
]

# 2. Split chronologically: train on 2023–2024, test on 2025
train_df = model_features_df[
    model_features_df["season"].isin([2023, 2024])
].copy()

test_df = model_features_df[
    model_features_df["season"] == 2025
].copy()

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))
print("Training seasons:", sorted(train_df["season"].unique()))
print("Testing seasons:", sorted(test_df["season"].unique()))

# 3. Create the models
def make_model():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            max_iter=2000,
            random_state=42
        ))
    ])

baseline_model_v2 = make_model()
enhanced_model = make_model()

# 4. Train
baseline_model_v2.fit(
    train_df[baseline_features],
    train_df["won_race"]
)

enhanced_model.fit(
    train_df[enhanced_features],
    train_df["won_race"]
)

# 5. Predict each driver's chance of winning
test_df["baseline_score"] = baseline_model_v2.predict_proba(
    test_df[baseline_features]
)[:, 1]

test_df["enhanced_score"] = enhanced_model.predict_proba(
    test_df[enhanced_features]
)[:, 1]

# 6. Rank drivers within each Grand Prix
test_df["baseline_rank"] = (
    test_df.groupby(["season", "round"])["baseline_score"]
    .rank(method="first", ascending=False)
)

test_df["enhanced_rank"] = (
    test_df.groupby(["season", "round"])["enhanced_score"]
    .rank(method="first", ascending=False)
)

# 7. Evaluate winner prediction at race level
def evaluate_race_predictions(df, rank_column):
    race_results = []

    for (season, round_number), race in df.groupby(["season", "round"]):
        race = race.copy()
        actual_winners = race[race["won_race"] == 1]

        if actual_winners.empty:
            continue

        winner = actual_winners.iloc[0]
        predicted = race.sort_values(rank_column).iloc[0]

        race_results.append({
            "season": season,
            "round": round_number,
            "actual_winner": winner["driver_name"],
            "predicted_winner": predicted["driver_name"],
            "top1_correct": int(predicted["won_race"] == 1),
            "winner_rank": int(winner[rank_column]),
            "winner_in_top3": int(winner[rank_column] <= 3)
        })

    return pd.DataFrame(race_results)

baseline_eval = evaluate_race_predictions(test_df, "baseline_rank")
enhanced_eval = evaluate_race_predictions(test_df, "enhanced_rank")

# 8. Compare results
comparison = pd.DataFrame([
    {
        "model": "Baseline: qualifying only",
        "races_evaluated": len(baseline_eval),
        "top1_correct": baseline_eval["top1_correct"].sum(),
        "top1_accuracy_pct": round(
            baseline_eval["top1_correct"].mean() * 100, 2
        ),
        "winner_in_top3_pct": round(
            baseline_eval["winner_in_top3"].mean() * 100, 2
        ),
        "mean_actual_winner_rank": round(
            baseline_eval["winner_rank"].mean(), 2
        )
    },
    {
        "model": "Enhanced: historical features",
        "races_evaluated": len(enhanced_eval),
        "top1_correct": enhanced_eval["top1_correct"].sum(),
        "top1_accuracy_pct": round(
            enhanced_eval["top1_correct"].mean() * 100, 2
        ),
        "winner_in_top3_pct": round(
            enhanced_eval["winner_in_top3"].mean() * 100, 2
        ),
        "mean_actual_winner_rank": round(
            enhanced_eval["winner_rank"].mean(), 2
        )
    }
])

print("\nMODEL COMPARISON — UNSEEN 2025 SEASON")
display(comparison)

# 9. Inspect individual race predictions
race_comparison = baseline_eval.merge(
    enhanced_eval,
    on=["season", "round"],
    suffixes=("_baseline", "_enhanced")
)

display(
    race_comparison[
        [
            "round",
            "actual_winner_baseline",
            "predicted_winner_baseline",
            "predicted_winner_enhanced",
            "winner_rank_baseline",
            "winner_rank_enhanced"
        ]
    ].sort_values("round")
)

NameError: name 'model_features_df' is not defined

In [ ]:

import pandas as pd
import numpy as np
from IPython.display import display

# Extract coefficients from the trained enhanced model
classifier = enhanced_model.named_steps["classifier"]

coefficient_table = pd.DataFrame({
    "feature": enhanced_features,
    "coefficient": classifier.coef_[0]
})

coefficient_table["absolute_effect"] = (
    coefficient_table["coefficient"].abs()
)

coefficient_table = coefficient_table.sort_values(
    "absolute_effect", ascending=False
)

print("ENHANCED MODEL: LEARNED FEATURE COEFFICIENTS")
display(coefficient_table)

# Compare the model's top prediction with the actual winner
diagnostic = test_df.copy()

diagnostic["qualifying_position_numeric"] = pd.to_numeric(
    diagnostic["qualifying_position_feature"],
    errors="coerce"
)

diagnostic["qualified_first"] = (
    diagnostic["qualifying_position_numeric"] == 1
)

print("\nPREDICTION CHECKS")
print(
    "2025 records:",
    len(diagnostic)
)
print(
    "Records with missing historical driver average:",
    diagnostic["driver_prior_mean_finish"].isna().sum()
)
print(
    "Records with missing historical constructor average:",
    diagnostic["constructor_prior_mean_finish"].isna().sum()
)

# Inspect races where the two models disagree
disagreements = race_comparison[
    race_comparison["predicted_winner_baseline"]
    != race_comparison["predicted_winner_enhanced"]
]

print("\nRACES WHERE THE MODELS DISAGREE")
display(
    disagreements[
        [
            "round",
            "actual_winner_baseline",
            "predicted_winner_baseline",
            "predicted_winner_enhanced",
            "winner_rank_baseline",
            "winner_rank_enhanced"
        ]
    ].sort_values("round")
)

NameError: name 'enhanced_model' is not defined

In [ ]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import log_loss
import pandas as pd
import numpy as np
from IPython.display import display

# Train a Random Forest using the same chronological split
rf_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", RandomForestClassifier(
        n_estimators=400,
        min_samples_leaf=10,
        max_features="sqrt",
        class_weight=None,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(
    train_df[enhanced_features],
    train_df["won_race"]
)

# Predict driver-level scores for the 2025 races
rf_test = test_df.copy()

rf_test["rf_score"] = rf_model.predict_proba(
    rf_test[enhanced_features]
)[:, 1]

rf_test["rf_rank"] = (
    rf_test.groupby(["season", "round"])["rf_score"]
    .rank(method="first", ascending=False)
)

rf_eval = evaluate_race_predictions(rf_test, "rf_rank")

rf_summary = pd.DataFrame([{
    "model": "Random Forest: historical features",
    "races_evaluated": len(rf_eval),
    "top1_correct": int(rf_eval["top1_correct"].sum()),
    "top1_accuracy_pct": round(
        rf_eval["top1_correct"].mean() * 100, 2
    ),
    "winner_in_top3_pct": round(
        rf_eval["winner_in_top3"].mean() * 100, 2
    ),
    "mean_actual_winner_rank": round(
        rf_eval["winner_rank"].mean(), 2
    )
}])

print("RANDOM FOREST — 2025 HOLDOUT RESULTS")
display(rf_summary)

# Compare all three models on the same races
all_model_comparison = pd.DataFrame([
    {
        "model": "Qualifying-only baseline",
        "top1_accuracy_pct": baseline_eval["top1_correct"].mean() * 100,
        "winner_in_top3_pct": baseline_eval["winner_in_top3"].mean() * 100,
        "mean_actual_winner_rank": baseline_eval["winner_rank"].mean()
    },
    {
        "model": "Logistic regression + historical features",
        "top1_accuracy_pct": enhanced_eval["top1_correct"].mean() * 100,
        "winner_in_top3_pct": enhanced_eval["winner_in_top3"].mean() * 100,
        "mean_actual_winner_rank": enhanced_eval["winner_rank"].mean()
    },
    {
        "model": "Random Forest + historical features",
        "top1_accuracy_pct": rf_eval["top1_correct"].mean() * 100,
        "winner_in_top3_pct": rf_eval["winner_in_top3"].mean() * 100,
        "mean_actual_winner_rank": rf_eval["winner_rank"].mean()
    }
])

all_model_comparison = all_model_comparison.round(2)
display(all_model_comparison)

# Show Random Forest predictions race by race
display(
    rf_eval[
        [
            "round",
            "actual_winner",
            "predicted_winner",
            "top1_correct",
            "winner_rank",
            "winner_in_top3"
        ]
    ].sort_values("round")
)

KeyError: "['driver_prior_mean_finish', 'driver_prior_win_rate', 'driver_recent5_mean_finish', 'constructor_prior_mean_finish', 'constructor_prior_win_rate'] not in index"

In [ ]:

# Audit the historical dataset before expanding training

audit = (
    model_features_df
    .groupby("season")
    .agg(
        driver_records=("driver_id", "size"),
        races=("round", "nunique"),
        winners=("won_race", "sum"),
        missing_qualifying=(
            "qualifying_position",
            lambda s: s.isna().sum()
        ),
        unique_drivers=("driver_id", "nunique")
    )
    .reset_index()
)

display(audit)

# Verify that each race has exactly one recorded winner
winner_check = (
    model_features_df
    .groupby(["season", "round"])["won_race"]
    .sum()
    .reset_index(name="winner_count")
)

print("Races checked:", len(winner_check))
print("Races with exactly one winner:",
      (winner_check["winner_count"] == 1).sum())
print("Races with zero or multiple winners:",
      (winner_check["winner_count"] != 1).sum())

NameError: name 'model_features_df' is not defined

In [ ]:

# Build a larger, chronological model dataset

expanded_df = history[
    history["season"].between(2018, 2025)
].copy()

expanded_df["starting_position_feature"] = pd.to_numeric(
    expanded_df["grid"], errors="coerce"
)

# Keep only rows with a known starting position
expanded_df = expanded_df[
    expanded_df["starting_position_feature"].notna()
].copy()

# Define the new, consistent feature sets
grid_baseline_features = ["starting_position_feature"]

expanded_features = [
    "starting_position_feature",
    "driver_prior_mean_finish",
    "driver_prior_win_rate",
    "driver_recent5_mean_finish",
    "constructor_prior_mean_finish",
    "constructor_prior_win_rate"
]

# Chronological split
expanded_train = expanded_df[
    expanded_df["season"].between(2018, 2024)
].copy()

expanded_test = expanded_df[
    expanded_df["season"] == 2025
].copy()

print("Expanded training rows:", len(expanded_train))
print("2025 test rows:", len(expanded_test))
print("Training seasons:",
      sorted(expanded_train["season"].unique()))
print("Testing seasons:",
      sorted(expanded_test["season"].unique()))

# Check race-level target integrity
winner_audit = (
    expanded_df.groupby(["season", "round"])["won_race"]
    .sum()
)

print("\nTotal races:", len(winner_audit))
print("Races with exactly one winner:",
      int((winner_audit == 1).sum()))
print("Races with invalid winner counts:",
      int((winner_audit != 1).sum()))

print("\nRows by season:")
display(
    expanded_df.groupby("season").agg(
        driver_records=("driver_id", "size"),
        races=("round", "nunique"),
        winners=("won_race", "sum"),
        missing_grid=("starting_position_feature",
                      lambda s: s.isna().sum())
    ).reset_index()
)


NameError: name 'history' is not defined

In [1]:

from pathlib import Path

PROJECT_DIR = Path(
    r"C:\Users\saksham\Desktop\Project\f1-prediction-platform"
)

# Check likely locations for the historical dataset
candidates = [
    PROJECT_DIR / "data" / "processed" / "expanded_model_dataset.csv",
    PROJECT_DIR / "data" / "processed" / "history.csv",
    PROJECT_DIR / "data" / "history.csv",
]

for path in candidates:
    print(f"{'FOUND' if path.exists() else 'NOT FOUND'}: {path}")


FOUND: C:\Users\saksham\Desktop\Project\f1-prediction-platform\data\processed\expanded_model_dataset.csv
NOT FOUND: C:\Users\saksham\Desktop\Project\f1-prediction-platform\data\processed\history.csv
NOT FOUND: C:\Users\saksham\Desktop\Project\f1-prediction-platform\data\history.csv


In [2]:

import pandas as pd

history_path = (
    PROJECT_DIR / "data" / "processed" / "expanded_model_dataset.csv"
)

history = pd.read_csv(history_path)

print("History restored:", history.shape)
print("Columns:", history.columns.tolist())
print("Seasons:", sorted(history["season"].dropna().unique()))


History restored: (3458, 27)
Columns: ['season', 'round', 'race_name', 'race_date', 'driver_id', 'driver_code', 'driver_number', 'driver_name', 'constructor_id', 'constructor_name', 'grid', 'position', 'position_text', 'points', 'laps', 'status', 'finish_time', 'fastest_lap_rank', 'fastest_lap_time', 'position_num', 'won_race', 'driver_prior_mean_finish', 'driver_prior_win_rate', 'driver_recent5_mean_finish', 'constructor_prior_mean_finish', 'constructor_prior_win_rate', 'starting_position_feature']
Seasons: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


In [3]:

import pandas as pd
from pathlib import Path

PROJECT_DIR = Path(
    r"C:\Users\saksham\Desktop\Project\f1-prediction-platform"
)

dataset_path = (
    PROJECT_DIR / "data" / "processed" / "expanded_model_dataset.csv"
)

expanded_df = pd.read_csv(dataset_path)

print("Dataset loaded successfully")
print("Shape:", expanded_df.shape)

print("\nColumns:")
print(expanded_df.columns.tolist())

print("\nSeasons available:")
print(sorted(expanded_df["season"].dropna().unique()))

print("\nRequired-column audit:")
required_columns = [
    "season",
    "round",
    "driver_id",
    "won_race",
    "grid",
    "driver_prior_mean_finish",
    "driver_prior_win_rate",
    "driver_recent5_mean_finish",
    "constructor_prior_mean_finish",
    "constructor_prior_win_rate",
]

for column in required_columns:
    status = "FOUND" if column in expanded_df.columns else "MISSING"
    print(f"{status}: {column}")

if "grid" in expanded_df.columns:
    print("\nGrid missing values:", expanded_df["grid"].isna().sum())

if {"season", "round", "won_race"}.issubset(expanded_df.columns):
    winner_audit = expanded_df.groupby(["season", "round"])["won_race"].sum()
    print("\nRace groups:", len(winner_audit))
    print("Groups with exactly one winner:", int((winner_audit == 1).sum()))
    print("Groups with invalid winner counts:", int((winner_audit != 1).sum()))

print("\nRows by season:")
if "season" in expanded_df.columns:
    print(expanded_df.groupby("season").size().to_string())


Dataset loaded successfully
Shape: (3458, 27)

Columns:
['season', 'round', 'race_name', 'race_date', 'driver_id', 'driver_code', 'driver_number', 'driver_name', 'constructor_id', 'constructor_name', 'grid', 'position', 'position_text', 'points', 'laps', 'status', 'finish_time', 'fastest_lap_rank', 'fastest_lap_time', 'position_num', 'won_race', 'driver_prior_mean_finish', 'driver_prior_win_rate', 'driver_recent5_mean_finish', 'constructor_prior_mean_finish', 'constructor_prior_win_rate', 'starting_position_feature']

Seasons available:
[np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Required-column audit:
FOUND: season
FOUND: round
FOUND: driver_id
FOUND: won_race
FOUND: grid
FOUND: driver_prior_mean_finish
FOUND: driver_prior_win_rate
FOUND: driver_recent5_mean_finish
FOUND: constructor_prior_mean_finish
FOUND: constructor_prior_win_rate

Grid missing values: 0

Race groups: 173
Groups with exactly one 

In [5]:

# Create the chronological train/test split

expanded_features = [
    "starting_position_feature",
    "driver_prior_mean_finish",
    "driver_prior_win_rate",
    "driver_recent5_mean_finish",
    "constructor_prior_mean_finish",
    "constructor_prior_win_rate",
]

target = "won_race"

# Training data: 2018–2024
expanded_train = expanded_df[
    expanded_df["season"].between(2018, 2024)
].copy()

# Unseen test data: 2025
expanded_test = expanded_df[
    expanded_df["season"] == 2025
].copy()

print("Training shape:", expanded_train.shape)
print("Testing shape:", expanded_test.shape)

print("\nTraining seasons:",
      sorted(expanded_train["season"].unique()))

print("Testing seasons:",
      sorted(expanded_test["season"].unique()))

print("\nTraining rows by season:")
print(expanded_train.groupby("season").size().to_string())

print("\nTest rows by season:")
print(expanded_test.groupby("season").size().to_string())

# Validate features and target
assert all(c in expanded_df.columns for c in expanded_features)
assert expanded_train[target].notna().all()
assert expanded_test[target].notna().all()
assert expanded_train["season"].max() < expanded_test["season"].min()

print("\nChronological split created successfully.")


Training shape: (2979, 27)
Testing shape: (479, 27)

Training seasons: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Testing seasons: [np.int64(2025)]

Training rows by season:
season
2018    420
2019    420
2020    340
2021    440
2022    440
2023    440
2024    479

Test rows by season:
season
2025    479

Chronological split created successfully.


In [6]:

# Audit engineered features before model training

prior_features = [
    "driver_prior_mean_finish",
    "driver_prior_win_rate",
    "driver_recent5_mean_finish",
    "constructor_prior_mean_finish",
    "constructor_prior_win_rate",
]

print("=== FEATURE LEAKAGE PRE-CHECK ===")

# 1. Missing values
print("\nMissing values in historical features:")
print(expanded_df[prior_features].isna().sum())

# 2. Win-rate ranges
rate_features = [
    "driver_prior_win_rate",
    "constructor_prior_win_rate",
]

print("\nWin-rate feature ranges:")
for col in rate_features:
    print(
        f"{col}: min={expanded_df[col].min()}, "
        f"max={expanded_df[col].max()}"
    )

    assert expanded_df[col].dropna().between(0, 1).all(), (
        f"{col} contains values outside [0, 1]"
    )

# 3. Finish-position ranges
finish_features = [
    "driver_prior_mean_finish",
    "driver_recent5_mean_finish",
    "constructor_prior_mean_finish",
]

print("\nHistorical finish-position ranges:")
for col in finish_features:
    print(
        f"{col}: min={expanded_df[col].min()}, "
        f"max={expanded_df[col].max()}"
    )

# 4. Inspect the first chronological records
print("\nEarliest records for manual inspection:")
display(
    expanded_df.sort_values(
        ["season", "round", "driver_id"]
    )[
        [
            "season",
            "round",
            "race_name",
            "driver_name",
            "constructor_name",
            "position_num",
        ] + prior_features
    ].head(15)
)


=== FEATURE LEAKAGE PRE-CHECK ===

Missing values in historical features:
driver_prior_mean_finish         43
driver_prior_win_rate            43
driver_recent5_mean_finish       43
constructor_prior_mean_finish    32
constructor_prior_win_rate       32
dtype: int64

Win-rate feature ranges:
driver_prior_win_rate: min=0.0, max=1.0
constructor_prior_win_rate: min=0.0, max=1.0

Historical finish-position ranges:
driver_prior_mean_finish: min=1.0, max=20.0
driver_recent5_mean_finish: min=1.0, max=20.0
constructor_prior_mean_finish: min=2.0, max=17.0

Earliest records for manual inspection:


,season,round,race_name,driver_name,constructor_name,position_num,driver_prior_mean_finish,driver_prior_win_rate,driver_recent5_mean_finish,constructor_prior_mean_finish,constructor_prior_win_rate
0,2018,1,Australian Grand Prix,Fernando Alonso,McLaren,5,NaN,NaN,NaN,NaN,NaN
1,2018,1,Australian Grand Prix,Valtteri Bottas,Mercedes,8,NaN,NaN,NaN,NaN,NaN
2,2018,1,Australian Grand Prix,Brendon Hartley,Toro Rosso,15,NaN,NaN,NaN,NaN,NaN
3,2018,1,Australian Grand Prix,Marcus Ericsson,Sauber,19,NaN,NaN,NaN,NaN,NaN
4,2018,1,Australian Grand Prix,Pierre Gasly,Toro Rosso,18,NaN,NaN,NaN,NaN,NaN
5,2018,1,Australian Grand Prix,Romain Grosjean,Haas F1 Team,16,NaN,NaN,NaN,NaN,NaN
6,2018,1,Australian Grand Prix,Lewis Hamilton,Mercedes,2,NaN,NaN,NaN,NaN,NaN
7,2018,1,Australian Grand Prix,Nico Hülkenberg,Renault,7,NaN,NaN,NaN,NaN,NaN
8,2018,1,Australian Grand Prix,Kevin Magnussen,Haas F1 Team,17,NaN,NaN,NaN,NaN,NaN
9,2018,1,Australian Grand Prix,Charles Leclerc,Sauber,13,NaN,NaN,NaN,NaN,NaN


In [7]:

from pathlib import Path
import json

PROJECT_DIR = Path(
    r"C:\Users\saksham\Desktop\Project\f1-prediction-platform"
)

search_terms = [
    "driver_prior_mean_finish",
    "driver_recent5_mean_finish",
    "constructor_prior_mean_finish",
]

matches_found = 0

for path in PROJECT_DIR.rglob("*"):
    if not path.is_file():
        continue

    if any(
        part in {".git", ".venv", "venv", "__pycache__"}
        for part in path.parts
    ):
        continue

    if path.suffix == ".py":
        try:
            content = path.read_text(encoding="utf-8")
        except (UnicodeDecodeError, OSError):
            continue

        lines = content.splitlines()
        matching_lines = [
            i for i, line in enumerate(lines)
            if any(term in line for term in search_terms)
        ]

        if matching_lines:
            print(f"\n--- FILE: {path.name} ---")
            shown = set()

            for i in matching_lines:
                for j in range(max(0, i - 5), min(len(lines), i + 9)):
                    if j not in shown:
                        print(f"{j + 1}: {lines[j]}")
                        shown.add(j)
                print("...")
            matches_found += 1

    elif path.suffix == ".ipynb":
        try:
            notebook = json.loads(
                path.read_text(encoding="utf-8")
            )
        except (json.JSONDecodeError, OSError, UnicodeDecodeError):
            continue

        for cell_number, cell in enumerate(notebook.get("cells", []), 1):
            if cell.get("cell_type") != "code":
                continue

            source = "".join(cell.get("source", []))

            if any(term in source for term in search_terms):
                print(
                    f"\n--- NOTEBOOK: {path.name}, "
                    f"CELL {cell_number} ---"
                )
                print(source[:5000])
                print("...")
                matches_found += 1

print(f"\nSearch complete. Matching files/cells: {matches_found}")



--- NOTEBOOK: 01_data_collection.ipynb, CELL 1 ---

from pathlib import Path
import pandas as pd
import joblib

# Project paths
PROJECT_DIR = Path(
    r"C:\Users\saksham\Desktop\Project\f1-prediction-platform"
)

DATA_PATH = (
    PROJECT_DIR / "data" / "processed"
    / "expanded_model_dataset.csv"
)

MODEL_DIR = PROJECT_DIR / "models"
REPORT_DIR = PROJECT_DIR / "reports"

# Load the saved dataset
expanded_df = pd.read_csv(DATA_PATH)

# Restore the feature list
expanded_features = [
    "starting_position_feature",
    "driver_prior_mean_finish",
    "driver_prior_win_rate",
    "driver_recent5_mean_finish",
    "constructor_prior_mean_finish",
    "constructor_prior_win_rate",
]

target = "won_race"

# Restore the saved models
baseline_model = joblib.load(
    MODEL_DIR / "baseline_model.joblib"
)

enhanced_model = joblib.load(
    MODEL_DIR / "logistic_regression_model.joblib"
)

rf_model = joblib.load(
    MODEL_DIR / "random_forest_model.joblib"
)

print("Project restored succes

In [9]:

from pathlib import Path

PROJECT_DIR = Path(
    r"C:\Users\saksham\Desktop\Project\f1-prediction-platform"
)

print("Project folder exists:", PROJECT_DIR.exists())

if PROJECT_DIR.exists():
    matches = list(PROJECT_DIR.rglob("01_data_collection.ipynb"))

    if matches:
        for path in matches:
            print("Notebook found:", path)
    else:
        print("Notebook not found inside the project folder.")
        print("\nNotebooks available in the project:")
        for path in PROJECT_DIR.rglob("*.ipynb"):
            print(path)
else:
    print("Check the project folder path.")
    print("\nCurrent Jupyter working directory:")
    print(Path.cwd())


Project folder exists: True
Notebook found: C:\Users\saksham\Desktop\Project\f1-prediction-platform\notebooks\01_data_collection.ipynb


In [10]:

import json
from pathlib import Path

PROJECT_DIR = Path(
    r"C:\Users\saksham\Desktop\Project\f1-prediction-platform"
)

notebook_path = PROJECT_DIR / "notebooks" / "01_data_collection.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

feature_names = [
    "driver_prior_mean_finish",
    "driver_prior_win_rate",
    "driver_recent5_mean_finish",
    "constructor_prior_mean_finish",
    "constructor_prior_win_rate",
]

keywords = [
    "groupby", "shift", "rolling", "expanding",
    "cumcount", "prior_mean", "prior_win_rate",
    "recent5_mean", "cumsum"
]

found = False

for i, cell in enumerate(notebook["cells"], start=1):
    if cell.get("cell_type") != "code":
        continue

    source = "".join(cell.get("source", []))
    lines = source.splitlines()

    # Print cells containing feature calculations or history operations.
    relevant = [
        j for j, line in enumerate(lines)
        if (
            any(name in line for name in feature_names)
            or any(word in line.lower() for word in keywords)
        )
    ]

    if not relevant:
        continue

    # Skip cells with no meaningful feature/history logic.
    if not any(
        any(name in line for name in feature_names)
        and any(word in line.lower() for word in keywords)
        for line in lines
    ):
        continue

    found = True
    print(f"\n{'=' * 60}")
    print(f"CELL {i}")
    print("=" * 60)

    shown = set()
    for j in relevant:
        for k in range(max(0, j - 5), min(len(lines), j + 7)):
            if k not in shown:
                print(f"{k + 1}: {lines[k]}")
                shown.add(k)
        print("...")

if not found:
    print("No matching feature-calculation code found in this notebook.")
    print("We may need to search the other notebooks or Python files.")



CELL 1
20: expanded_df = pd.read_csv(DATA_PATH)
21: 
22: # Restore the feature list
23: expanded_features = [
24:     "starting_position_feature",
25:     "driver_prior_mean_finish",
26:     "driver_prior_win_rate",
27:     "driver_recent5_mean_finish",
28:     "constructor_prior_mean_finish",
29:     "constructor_prior_win_rate",
30: ]
31: 
...
32: target = "won_race"
...
33: 
...
34: # Restore the saved models
...
35: baseline_model = joblib.load(
...

CELL 45
18: 
19: # ----------------------------------------
20: # A. DRIVER HISTORICAL FEATURES
21: # ----------------------------------------
22: 
23: driver_groups = history.groupby("driver_id", sort=False)
24: 
25: # Average finish in races BEFORE the current race
26: history["driver_prior_mean_finish"] = (
27:     driver_groups["position_num"]
28:     .transform(lambda s: s.shift(1).expanding(min_periods=1).mean())
29: )
...
30: 
31: # Win rate BEFORE the current race
32: history["driver_prior_win_rate"] = (
...
33:     driver_gro

In [11]:

from pathlib import Path

PROJECT_DIR = Path(
    r"C:\Users\saksham\Desktop\Project\f1-prediction-platform"
)

terms = [
    "driver_prior_mean_finish",
    "driver_prior_win_rate",
    "driver_recent5_mean_finish",
    "constructor_prior_mean_finish",
    "constructor_prior_win_rate",
]

for path in PROJECT_DIR.rglob("*"):
    if not path.is_file():
        continue

    if path.suffix.lower() not in {".py", ".ipynb", ".csv"}:
        continue

    if any(part.lower() in {
        ".git", ".venv", "venv", "__pycache__"
    } for part in path.parts):
        continue

    try:
        content = path.read_text(encoding="utf-8")
    except Exception:
        continue

    found = [term for term in terms if term in content]

    if found:
        print(f"\nFILE: {path.relative_to(PROJECT_DIR)}")
        print("Matching features:", ", ".join(found))



FILE: notebooks\01_data_collection.ipynb
Matching features: driver_prior_mean_finish, driver_prior_win_rate, driver_recent5_mean_finish, constructor_prior_mean_finish, constructor_prior_win_rate

FILE: notebooks\recovery.ipynb
Matching features: driver_prior_mean_finish, driver_prior_win_rate, driver_recent5_mean_finish, constructor_prior_mean_finish, constructor_prior_win_rate

FILE: data\processed\expanded_model_dataset.csv
Matching features: driver_prior_mean_finish, driver_prior_win_rate, driver_recent5_mean_finish, constructor_prior_mean_finish, constructor_prior_win_rate

FILE: data\processed\historical_model_features.csv
Matching features: driver_prior_mean_finish, driver_prior_win_rate, driver_recent5_mean_finish, constructor_prior_mean_finish, constructor_prior_win_rate


In [12]:

from pathlib import Path
import json
import pandas as pd

PROJECT_DIR = Path(
    r"C:\Users\saksham\Desktop\Project\f1-prediction-platform"
)

# 1. List all notebooks in the project.
print("=== PROJECT NOTEBOOKS ===")
for path in PROJECT_DIR.rglob("*.ipynb"):
    if not any(part.lower() in {
        ".git", ".venv", "venv", "__pycache__"
    } for part in path.parts):
        print(path.relative_to(PROJECT_DIR))

# 2. Inspect code cells that assign or calculate historical features.
print("\n=== FEATURE CALCULATION CELLS ===")

for notebook_path in PROJECT_DIR.rglob("*.ipynb"):
    if any(part.lower() in {
        ".git", ".venv", "venv", "__pycache__"
    } for part in notebook_path.parts):
        continue

    try:
        notebook = json.loads(notebook_path.read_text(encoding="utf-8"))
    except Exception:
        continue

    for i, cell in enumerate(notebook.get("cells", []), start=1):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))
        if (
            any(name in source for name in [
                "driver_prior_mean_finish",
                "driver_prior_win_rate",
                "driver_recent5_mean_finish",
                "constructor_prior_mean_finish",
                "constructor_prior_win_rate",
            ])
            and any(word in source.lower() for word in [
                "groupby", "shift(", "rolling(", "expanding(",
                "cumsum(", "transform(", "merge("
            ])
        ):
            print(
                f"\nNotebook: {notebook_path.relative_to(PROJECT_DIR)}"
                f"\nCell: {i}\n"
            )
            print(source[:5000])
            print("\n--- END CELL ---")

# 3. Inspect the historical dataset's columns and sample rows.
csv_path = PROJECT_DIR / "data" / "processed" / "historical_model_features.csv"

print("\n=== HISTORICAL DATASET ===")
if csv_path.exists():
    df = pd.read_csv(csv_path)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("\nFirst 3 rows:")
    print(df.head(3).to_string(index=False))
else:
    print("Historical dataset not found.")


=== PROJECT NOTEBOOKS ===
notebooks\01_data_collection.ipynb
notebooks\recovery.ipynb

=== FEATURE CALCULATION CELLS ===

Notebook: notebooks\01_data_collection.ipynb
Cell: 45


import pandas as pd

# Start from the complete historical race-results dataset
history = results_df.copy()

history["position_num"] = pd.to_numeric(
    history["position"], errors="coerce"
)
history["won_race"] = (
    history["position_num"] == 1
).astype(int)

# Sort chronologically before calculating historical features
history = history.sort_values(
    ["season", "round", "driver_id"]
).reset_index(drop=True)

# ----------------------------------------
# A. DRIVER HISTORICAL FEATURES
# ----------------------------------------

driver_groups = history.groupby("driver_id", sort=False)

# Average finish in races BEFORE the current race
history["driver_prior_mean_finish"] = (
    driver_groups["position_num"]
    .transform(lambda s: s.shift(1).expanding(min_periods=1).mean())
)

# Win rate BEFORE the current

In [2]:

import numpy as np
import pandas as pd

score_columns = {
    "Baseline": "baseline_score",
    "Logistic Regression": "logistic_score",
    "Random Forest": "random_forest_score"
}

normalized_predictions = test_predictions.copy()
probability_columns = {}

for model_name, score_col in score_columns.items():
    probability_col = (
        model_name.lower()
        .replace(" ", "_")
        .replace("regression", "regression")
        + "_normalized_probability"
    )

    score_sums = normalized_predictions.groupby(
        ["season", "round"]
    )[score_col].transform("sum")

    normalized_predictions[probability_col] = (
        normalized_predictions[score_col] / score_sums
    )

    probability_columns[model_name] = probability_col

# Evaluate each race's actual winner probability
evaluation_rows = []

for (season, round_no), race in normalized_predictions.groupby(
    ["season", "round"]
):
    actual = race["won_race"].to_numpy(dtype=float)

    for model_name, probability_col in probability_columns.items():
        probabilities = race[probability_col].to_numpy(dtype=float)
        probabilities = np.clip(probabilities, 1e-15, 1.0)
        winner_probability = probabilities[actual == 1][0]

        evaluation_rows.append({
            "season": season,
            "round": round_no,
            "model": model_name,
            "winner_probability": winner_probability,
            "log_loss": -np.log(winner_probability),
            "brier_score": np.sum((probabilities - actual) ** 2)
        })

normalized_metrics = pd.DataFrame(evaluation_rows)

normalized_summary = (
    normalized_metrics.groupby("model")
    .agg(
        mean_log_loss=("log_loss", "mean"),
        mean_brier_score=("brier_score", "mean"),
        mean_winner_probability=("winner_probability", "mean")
    )
    .round(4)
    .sort_values("mean_log_loss")
)

display(normalized_summary)

# Verify that each race's probabilities sum to 1
display(
    normalized_predictions.groupby(["season", "round"])[
        list(probability_columns.values())
    ].sum().head()
)


NameError: name 'test_predictions' is not defined

In [40]:


# Read-only audit: inspect starting-position values by season
position_audit = (
    expanded_df.groupby("season")
    .agg(
        rows=("starting_position_feature", "size"),
        missing_positions=("starting_position_feature",
                           lambda s: s.isna().sum()),
        unique_positions=("starting_position_feature",
                          "nunique"),
        minimum_position=("starting_position_feature", "min"),
        maximum_position=("starting_position_feature", "max"),
    )
    .reset_index()
)

display(position_audit)

# Compare the feature against the raw grid column, if available
if "grid" in expanded_df.columns:
    print(
        "Feature equals grid:",
        expanded_df["starting_position_feature"]
        .equals(expanded_df["grid"])
    )
    display(
        expanded_df[
            ["season", "round", "driver_name",
             "grid", "starting_position_feature"]
        ].head(10)
    )
else:
    print("No raw 'grid' column exists in expanded_df.")


,season,rows,missing_positions,unique_positions,minimum_position,maximum_position
0,2018,420,1,20,1.0,20.0
1,2019,420,12,20,1.0,20.0
2,2020,340,5,20,1.0,20.0
3,2021,440,12,20,1.0,20.0
4,2022,440,11,20,1.0,20.0
5,2023,440,3,20,1.0,20.0
6,2024,479,1,20,1.0,20.0
7,2025,479,0,20,1.0,20.0


Feature equals grid: False


,season,round,driver_name,grid,starting_position_feature
0,2018,1,Fernando Alonso,10,10.0
1,2018,1,Valtteri Bottas,15,15.0
2,2018,1,Brendon Hartley,16,16.0
3,2018,1,Marcus Ericsson,17,17.0
4,2018,1,Pierre Gasly,20,20.0
5,2018,1,Romain Grosjean,6,6.0
6,2018,1,Lewis Hamilton,1,1.0
7,2018,1,Nico Hülkenberg,7,7.0
8,2018,1,Kevin Magnussen,5,5.0
9,2018,1,Charles Leclerc,18,18.0
